In [2]:
# %pip install -q --force-reinstall --no-cache-dir \
#   "numpy==1.26.4" \
#   "scipy==1.13.1" \
#   "pandas==2.2.2" \
#   "datasets==3.6.0" \
#   "transformers==4.55.4" \
#   "accelerate==1.9.0" \
#   "tokenizers>=0.21,<0.22" \
#   "huggingface_hub>=0.30,<1.0" \
#   "lettucedetect==0.1.3"

In [3]:
# import numpy, scipy, pandas, transformers, accelerate
# print("numpy:", numpy.__version__)
# print("scipy:", scipy.__version__)
# print("pandas:", pandas.__version__)
# print("transformers:", transformers.__version__)
# print("accelerate:", accelerate.__version__)

# from lettucedetect.models.inference import HallucinationDetector
# print("LettuceDetect import OK")

# Hallucination Detection in Tool Calling
Self-contained final notebook for the course assignment.


## Assignment
Hallucination Detection in Tool Calling.


## Student name
Margarita Urazmetova


## CodaLab / username
project without codalab (marrita)


## Additional comments

This notebook is self-contained: it does **not** install the GitHub repository. Instead, it writes all project modules directly from notebook cells, builds a ToolACE-derived synthetic hallucination dataset, evaluates assignment baselines, trains the improved detector, and prints the main result tables in the notebook output.

Compared with earlier iterations, this version makes two important changes:

1. The corruption pipeline is harder and less template-like: contradictions include number/date/status/entity/value substitutions; overgeneration uses varied unsupported claims; missing-tool hallucinations are conditioned on unavailable tools.
2. The LookBackLens baseline is no longer only a lexical proxy. The notebook implements an **adapted attention-based LookBackLens** baseline using teacher-forced attention maps from a small causal language model. This is still an adaptation to a fixed-answer benchmark, but it uses the core LookBackLens signal: how much answer-token attention goes back to the context.

The notebook also keeps the real LettuceDetect baseline slot. If real LettuceDetect cannot be loaded, it is reported as unavailable rather than silently replaced by a keyword fallback.


# 2. Technical Report


## 2.1 Current methodology

The assignment asks us to build a dataset for hallucination detection in tool-calling dialogues, evaluate baseline methods, and improve their results. The required dataset construction is synthetic by design: starting from ToolACE-style dialogues, we automatically inject three hallucination types and store span-level labels in a RAGTruth-like format.

This final notebook follows the current methodology:

1. **Dataset construction.** We load ToolACE records containing a user query, available tools, a tool call, a tool output, and the final model answer. For each base record we keep one clean example and create three corrupted variants: `tool_contradiction`, `overgeneration`, and `missing_tool`.
2. **Harder hallucination injection.** To avoid an overly easy benchmark, the corruption module uses several styles: numeric/date/time substitutions, status/value flips, entity replacements, cross-field value changes, unsupported causal/history/future claims, and missing-tool actions conditioned on unavailable tools.
3. **RAGTruth-like supervision.** Each corrupted record stores `query`, `context` (tool output), `output` (final answer), and `hallucination_labels` / `labels` with exact character offsets of injected hallucinated spans.
4. **Validation checks.** We verify required fields, span offsets, clean/corrupted label consistency, corruption style diversity, class balance, and absence of `source_id` leakage between train/validation/test.
5. **Baselines.** We evaluate trivial baselines, lexical/rule baselines, an NLI verifier, real LettuceDetect if available, and an adapted attention-based LookBackLens baseline.
6. **Improved method.** We train a class-weighted ModernBERT token classifier. It receives query + tool output + answer, but loss is applied only to answer tokens. Tokens overlapping hallucinated spans are positive, other answer tokens are negative, and query/context tokens are masked.
7. **Metrics.** The main ranking-style evaluation is binary sentence/sample-level detection. Because positives are frequent, we focus on `sentence_macro_f1`, `balanced_accuracy`, confusion matrix, ROC-AUC and PR-AUC. Span-level metrics are reported separately for localization.


The improvement stage is framed as an improvement of the BERT-based span-detection baseline family represented by LettuceDetect. LettuceDetect is evaluated as a general hallucination detector, while our ModernBERT token classifier refines the same span-detection idea for the ToolACE-derived tool-calling setting using task-specific supervision, class-weighted token loss, and validation-tuned sentence/span thresholds.


## 2.2 Development history and previous results

Before this final version, we tried several simpler variants:

- **Initial implementation.** The first full Kaggle run failed during token-classifier training because `offset_mapping` was left inside the tokenized dataset and `DataCollatorForTokenClassification` attempted to pad nested offset lists. The final code fixes this by popping `offset_mapping` before collation.
- **Simple synthetic corruptions.** Early corruptions were more template-like, especially for overgeneration and missing-tool hallucinations. This produced strong numbers but risked testing whether the detector learned templates rather than factual grounding.
- **Fallback baselines.** Earlier notebooks allowed `lettucedetect_fallback_keyword`, which was useful for debugging but is not a valid replacement for the assignment LettuceDetect baseline. The final baseline table separates real LettuceDetect from unavailable/fallback cases.
- **LookBackLens-style proxy.** We first used lexical/context-overlap support features as a lightweight LookBackLens-style proxy. This is useful as a sanity baseline but does not measure attention during generation. The final version adds an adapted attention-based LookBackLens implementation.

In the previous executed run, the ModernBERT token classifier achieved approximately `sentence_macro_f1 = 0.824`, `balanced_accuracy = 0.878`, `ROC-AUC = 0.922`, `PR-AUC = 0.977`, and `span_char_f1 = 0.919`. It outperformed the strongest non-improved baseline in that run, but the per-type analysis showed that `tool_contradiction` remained the hardest category. The present notebook is meant to re-run the experiment on a harder dataset and with a more faithful attention-based LookBackLens baseline, so the final numbers may change.


## 2.3 How to interpret the results

A high positive-class F1 alone is not sufficient evidence of a good hallucination detector. In this dataset, the positive class can be around 75% because each clean example has three corrupted variants. An `always_hallucinated` classifier may therefore achieve a deceptively high positive-class F1 while failing to recognize clean answers.

For this reason, the final discussion should focus on:

- **`sentence_macro_f1`**: treats clean and hallucinated classes symmetrically;
- **`sentence_balanced_accuracy`**: averages recall for clean and hallucinated classes;
- **confusion matrix**: checks whether the model is not simply predicting one class;
- **ROC-AUC / PR-AUC**: evaluate score quality before a fixed threshold;
- **span metrics**: evaluate whether the detector localizes hallucinated text, not just detects it at the sample level;
- **per-type recall and span-F1**: show which hallucination types are easy or hard.

The adapted attention-based LookBackLens baseline should be interpreted as an adaptation, not an exact reproduction of the original paper setup. The original method collects attention during model generation; here, because the benchmark contains fixed corrupted answers, we use teacher-forced attention over the constructed `question + context + answer` sequence.


## 2.4 What the result section should prove

After `Runtime → Run all`, the results should establish the following:

- The improved ModernBERT detector beats trivial baselines such as `always_clean` and `always_hallucinated` on `sentence_macro_f1` and `balanced_accuracy`.
- Real LettuceDetect is either evaluated as `lettucedetect_real` or explicitly reported as unavailable. It is not silently replaced by a keyword method.
- The LookBackLens slot is evaluated using attention-based teacher-forced lookback signals, while the older lexical proxy is kept only as an additional sanity baseline if present.
- Span-level metrics are reported separately from sentence-level metrics.
- Per-type analysis explains whether the model handles all three required hallucination types or mainly succeeds on appended/template-like hallucinations.


### Hugging Face artifacts

The constructed ToolACE-derived hallucination dataset was published on Hugging Face:

`marrita/toolace-tool-calling-hallucination-ragtruth`

The trained ModernBERT token classifier was published on Hugging Face:

`marrita/modernbert-tool-calling-hallucination-detector`

## 2.5 Results and discussion

### Main evaluation setting

The main evaluation is **binary sentence-level hallucination detection**. Clean answers are treated as negative examples, while the three corrupted variants are treated as positive examples:

- `tool_contradiction`
- `overgeneration`
- `missing_tool`

Span labels are still used for training and localization. Therefore, the improved model is evaluated in two ways:

1. **Sentence/sample level:** whether the whole answer contains a hallucination.
2. **Span level:** whether the model localizes the hallucinated part of the answer.

This distinction is important because sentence-level detection is likely to be the main ranking-style metric, while span-level detection is still required by the assignment.

### Why macro-F1 and balanced accuracy are the main metrics

The dataset is intentionally imbalanced: for each clean ToolACE-derived example, we create several corrupted examples. In the final test split, 90 out of 120 examples are hallucinated, so the positive class rate is `0.75`.

Because of this, ordinary positive-class F1 can be misleading. For example, the `always_hallucinated` baseline predicts every answer as hallucinated. It obtains a high positive-class F1 (`0.857`) but completely fails on clean examples (`TN = 0`, `FP = 30`) and has only `0.500` balanced accuracy.

Therefore, the main metrics are:

- **sentence macro-F1** — averages F1 over clean and hallucinated classes;
- **sentence balanced accuracy** — averages recall over the two classes;
- **confusion matrix** — checks whether the model predicts both classes;
- **ROC-AUC / PR-AUC** — evaluate score quality before fixing a threshold;
- **span character F1** — measures how well hallucinated spans are localized;
- **relaxed span F1** — gives credit for overlapping span predictions;
- **per-type recall and span-F1** — show which hallucination types are easier or harder.

### Baselines

The notebook evaluates the following baselines:

| Method | Role |
|---|---|
| `always_clean` | trivial negative baseline |
| `always_hallucinated` | trivial positive baseline |
| `random_balanced` / `random_prior` | random sanity baselines |
| `keyword_tool_action` | rule-based baseline |
| `nli_sentence_verifier` | semantic/NLI baseline |
| `lettucedetect_real` | required LettuceDetect baseline |
| `attention_lookback_lens_adapted` | adapted attention-based LookBackLens baseline |
| `lookback_lens_style_proxy` | older lightweight proxy, kept only as an auxiliary sanity baseline |

The required baselines from the assignment are represented by `lettucedetect_real` and `attention_lookback_lens_adapted`. The LookBackLens implementation is adapted because the original method assumes access to generation-time attention traces. In this fixed-answer benchmark, we instead use teacher-forced attention maps over the constructed `question + context + answer` sequence. This still uses the central LookBackLens signal: how much answer-token attention goes back to the context.

### Improved method

The improved method is `modernbert_token_classifier`. It is best understood as an improvement of the LettuceDetect-style BERT span-detection baseline.

Instead of applying a general-purpose detector out of the box, we train a task-specific ModernBERT token classifier on the ToolACE-derived span labels. The input contains the user query, tool output, and final answer. The loss is applied only to answer tokens:

- hallucinated answer tokens are positive;
- non-hallucinated answer tokens are negative;
- query and context tokens are masked from the loss.

The model also uses class-weighted token-level training, because hallucinated tokens are much rarer than normal answer tokens. Sentence-level and span-level thresholds are tuned separately on the validation split.

### Full sentence-level and span-level results

| Method | Accuracy | Balanced Acc | Precision | Recall | Positive F1 | Negative F1 | Macro-F1 | ROC-AUC | PR-AUC | Span char-F1 | Relaxed span-F1 |
|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|
| `modernbert_token_classifier` | **0.808** | **0.861** | **0.986** | 0.756 | 0.855 | **0.716** | **0.786** | **0.895** | **0.969** | **0.957** | **0.778** |
| `nli_sentence_verifier` | 0.767 | 0.633 | 0.810 | **0.900** | 0.853 | 0.440 | 0.646 | 0.654 | 0.809 | 0.211 | 0.197 |
| `keyword_tool_action` | 0.642 | 0.583 | 0.797 | 0.700 | 0.746 | 0.394 | 0.570 | 0.666 | 0.846 | 0.224 | 0.241 |
| `lookback_lens_style_proxy` | 0.625 | 0.561 | 0.785 | 0.689 | 0.734 | 0.366 | 0.550 | 0.643 | 0.840 | 0.388 | 0.360 |
| `lettucedetect_real` | 0.592 | 0.550 | 0.781 | 0.633 | 0.699 | 0.364 | 0.532 | 0.534 | 0.775 | 0.157 | 0.102 |
| `attention_lookback_lens_adapted` | 0.567 | 0.533 | 0.771 | 0.600 | 0.675 | 0.350 | 0.513 | 0.555 | 0.779 | 0.106 | 0.000 |
| `always_hallucinated` | 0.750 | 0.500 | 0.750 | **1.000** | **0.857** | 0.000 | 0.429 | 0.500 | 0.750 | 0.136 | 0.010 |
| `random_balanced` | 0.658 | 0.450 | 0.729 | 0.867 | 0.792 | 0.047 | 0.419 | 0.465 | 0.750 | 0.119 | 0.000 |
| `random_prior` | 0.483 | 0.422 | 0.700 | 0.544 | 0.613 | 0.225 | 0.419 | 0.372 | 0.703 | 0.125 | 0.013 |
| `always_clean` | 0.250 | 0.500 | 0.000 | 0.000 | 0.000 | 0.400 | 0.200 | 0.500 | 0.750 | 0.000 | 0.000 |

### Confusion matrix interpretation

The confusion matrix confirms that the improved model is not simply exploiting class imbalance.

| Method | TP | FP | FN | TN | Recall | Specificity | Balanced Acc | Macro-F1 |
|---|---:|---:|---:|---:|---:|---:|---:|---:|
| `modernbert_token_classifier` | 68 | **1** | 22 | **29** | 0.756 | **0.967** | **0.861** | **0.786** |
| `nli_sentence_verifier` | 81 | 19 | 9 | 11 | **0.900** | 0.367 | 0.633 | 0.646 |
| `lettucedetect_real` | 57 | 16 | 33 | 14 | 0.633 | 0.467 | 0.550 | 0.532 |
| `attention_lookback_lens_adapted` | 54 | 16 | 36 | 14 | 0.600 | 0.467 | 0.533 | 0.513 |
| `always_hallucinated` | **90** | 30 | **0** | 0 | **1.000** | 0.000 | 0.500 | 0.429 |

`always_hallucinated` has perfect recall but zero specificity. In contrast, ModernBERT has only one false positive on clean examples and much higher balanced accuracy. Its main remaining weakness is false negatives, especially in `tool_contradiction`.

### Comparison with baselines

The strongest non-improved auxiliary baseline is `nli_sentence_verifier` with `sentence_macro_f1 = 0.646`. The improved model reaches `0.786`, giving a gain of:

| Comparison | Macro-F1 delta |
|---|---:|
| ModernBERT vs. best auxiliary baseline (`nli_sentence_verifier`) | **+0.139** |
| ModernBERT vs. real LettuceDetect | **+0.254** |
| ModernBERT vs. adapted LookBackLens | **+0.273** |
| ModernBERT vs. always hallucinated | **+0.357** |

The improvement is even clearer for span localization. ModernBERT reaches `span_char_f1 = 0.957`, while the best non-improved span baseline is the old `lookback_lens_style_proxy` with `0.388`. Real LettuceDetect reaches `0.157` span character F1 on this constructed benchmark.

### Per-type behavior

The per-type analysis shows a clear pattern:

| Hallucination type | n | Recall | Specificity | Positive F1 | Span char-F1 | Relaxed span-F1 | Interpretation |
|---|---:|---:|---:|---:|---:|---:|---|
| `missing_tool` | 30 | **0.967** | 0.000 | 0.983 | **0.981** | **0.967** | easiest; often contains explicit unsupported actions |
| `overgeneration` | 30 | **0.967** | 0.000 | 0.983 | **0.978** | **0.900** | easy; hallucinated content is usually an added unsupported claim |
| `tool_contradiction` | 30 | **0.333** | 0.000 | 0.500 | **0.772** | **0.467** | hardest; often involves short value/entity/date/number substitutions |
| `clean` | 30 | 0.000 | **0.967** | 0.000 | 0.000 | 0.000 | used to measure false positives / specificity |

For per-type corrupted-only subsets, balanced accuracy and macro-F1 are less informative because each subset contains almost no negative examples. Recall and span metrics are more useful there.

The hardest category is `tool_contradiction`. This is expected because contradictions can be very short and subtle: a changed number, date, status, location, or entity may be enough to make the answer hallucinated. In contrast, overgeneration and missing-tool examples often introduce longer spans, making them easier to localize.

### Development history

Earlier versions of the notebook used simpler corruptions and a purely lexical LookBackLens-style proxy. In that easier setting, ModernBERT reached approximately `sentence_macro_f1 = 0.824`, `balanced_accuracy = 0.878`, `ROC-AUC = 0.922`, `PR-AUC = 0.977`, and `span_char_f1 = 0.919`. However, that benchmark was more template-like, so high scores were less informative.

The current version intentionally makes the synthetic corruption pipeline harder and more diverse. It also replaces the pure proxy with an adapted attention-based LookBackLens baseline and uses real LettuceDetect. As a result, some sentence-level numbers are lower, but the evaluation is more realistic and more convincing.

### Hugging Face artifacts

The constructed ToolACE-derived hallucination dataset was published on Hugging Face:

`marrita/toolace-tool-calling-hallucination-ragtruth`

The trained ModernBERT token classifier was published on Hugging Face:

`marrita/modernbert-tool-calling-hallucination-detector`

### Limitations

The dataset is synthetic by design, as required by the assignment. This is not a flaw: automatic corruption is how we obtain controlled span-level labels. However, synthetic datasets can contain artifacts from the corruption procedure. The results should therefore be interpreted as strong performance on this controlled ToolACE-derived benchmark, not as a guarantee of universal real-world hallucination detection.

The adapted LookBackLens baseline is not an exact reproduction of the original paper. It uses real attention maps, but in teacher-forcing mode over fixed corrupted answers rather than generation-time traces. This is more faithful than a lexical proxy, but it should still be described as an adaptation.

### Overall conclusion

The task-specific ModernBERT token classifier substantially improves over both required baselines and auxiliary baselines. The improvement is strongest in macro-F1, balanced accuracy, and span-level localization quality. The model is especially strong on `missing_tool` and `overgeneration`, while `tool_contradiction` remains the main source of errors and the most important direction for future improvement.

# 3. Code

All project code is included below. The notebook is designed for **Runtime → Run all** in Colab with a GPU runtime.


## 3.1 Requirements

**What happens here:** this cell installs runtime dependencies needed by the self-contained notebook. It intentionally does not install the GitHub repository, because all project code is written directly in later cells.


**Colab note.** If you previously ran a version of this notebook that used aggressive package upgrades and then got a NumPy/SciPy error, use `Runtime → Disconnect and delete runtime` or `Runtime → Restart runtime` before running this fixed notebook. The error is caused by a broken numerical-stack installation in the active Colab session, not by the ToolACE dataset itself.


**Explanation:** installs libraries used for data loading, metrics, Hugging Face models, and training. The install commands are deliberately conservative: they use `--upgrade-strategy only-if-needed` and do not force-upgrade Colab's numerical stack. This avoids the common NumPy/SciPy binary-mismatch error that can appear after aggressive package upgrades.


In [4]:
# No GitHub install is used. These are runtime dependencies only.
# IMPORTANT: do not use --force-reinstall and do not use `pip install -U` here.
# In Colab, aggressive upgrades can partially replace NumPy/SciPy and cause errors such as
# `ImportError: cannot import name '_center' from numpy._core.umath`.

!pip -q install --upgrade-strategy only-if-needed \
    "datasets>=2.18,<4" \
    "transformers>=4.48,<5" \
    "accelerate>=0.27,<2" \
    "scikit-learn>=1.2,<2" \
    "pandas>=2.0,<3" \
    "numpy>=1.24,<2.3" \
    "scipy>=1.10,<1.16" \
    "tqdm>=4.65" \
    "huggingface_hub>=0.24"

# Required external assignment baseline: real LettuceDetect.
# We install it conservatively instead of `lettucedetect -U` so that it does not upgrade NumPy/SciPy.
# If this fails because LettuceDetect needs an additional dependency, install only that missing package explicitly.
!pip -q install --upgrade-strategy only-if-needed lettucedetect

# Quick numerical-stack sanity check. If this cell fails after a previous broken install,
# restart the Colab runtime and run all cells from a clean session.
import numpy as np
import scipy
import pandas as pd
print('numpy', np.__version__, '| scipy', scipy.__version__, '| pandas', pd.__version__)


numpy 2.2.6 | scipy 1.13.1 | pandas 2.2.2


**Explanation:** verifies that the real `lettucedetect` package is importable. The detector model itself is loaded later during evaluation.

In [5]:
import importlib.util

if importlib.util.find_spec("lettucedetect") is None:
    raise RuntimeError("The lettucedetect package is not importable. Re-run the install cell or check Colab internet access.")
print("Real LettuceDetect package is importable.")


Real LettuceDetect package is importable.


**Explanation:** creates folders, sets random seeds, and prints the runtime/GPU information so the experiment is reproducible.


In [6]:
import os
import random
import sys
from pathlib import Path

import numpy as np
import pandas as pd

try:
    import torch
except Exception:
    torch = None

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
if torch is not None:
    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)

PROJECT_ROOT = Path.cwd()
PACKAGE_DIR = PROJECT_ROOT / "tool_hallucination_detection"
PACKAGE_DIR.mkdir(exist_ok=True)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Python:", sys.version)
print("Project root:", PROJECT_ROOT)
print("GPU available:", bool(torch is not None and torch.cuda.is_available()))
if torch is not None and torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
Project root: /content
GPU available: True
GPU: Tesla T4


## 3.2 Local package code

The following cells write a local `tool_hallucination_detection` package inside the notebook runtime.


**Explanation:** writes the schema utilities. These functions standardize records, validate span labels, and convert span labels into sentence-level binary labels.


In [7]:
%%writefile tool_hallucination_detection/schema.py
"""Shared record and label helpers for RAGTruth-style span annotations."""
from __future__ import annotations

from dataclasses import asdict, dataclass, field
from typing import Any, Mapping


@dataclass(frozen=True)
class SpanLabel:
    start: int
    end: int
    text: str
    label_type: str
    meta: str = ""

    def to_dict(self) -> dict[str, Any]:
        return asdict(self)


@dataclass
class HallucinationRecord:
    id: str
    source_id: str
    split: str
    corruption_type: str
    query: str
    tools: str
    tool_call: str
    context: str
    output: str
    labels: list[SpanLabel] = field(default_factory=list)

    def to_dict(self) -> dict[str, Any]:
        data = asdict(self)
        labels = [label.to_dict() for label in self.labels]
        data["labels"] = labels
        data["hallucination_labels"] = labels
        return data


def coerce_label(label: SpanLabel | Mapping[str, Any]) -> SpanLabel:
    if isinstance(label, SpanLabel):
        return label
    return SpanLabel(
        start=int(label["start"]),
        end=int(label["end"]),
        text=str(label["text"]),
        label_type=str(label["label_type"]),
        meta=str(label.get("meta", "")),
    )


def coerce_labels(labels: list[SpanLabel | Mapping[str, Any]] | None) -> list[SpanLabel]:
    return [coerce_label(label) for label in labels or []]


def sentence_label(record: Mapping[str, Any]) -> int:
    return int(len(record.get("labels") or []) > 0)


def validate_labels(record: Mapping[str, Any]) -> None:
    """Validate non-empty, non-overlapping spans and exact text offsets."""
    output = str(record.get("output", ""))
    labels = sorted(coerce_labels(record.get("labels")), key=lambda item: item.start)
    previous_end = -1
    for label in labels:
        if label.start < 0 or label.end <= label.start:
            raise ValueError(f"Invalid span bounds in record {record.get('id')}: {label}")
        if label.end > len(output):
            raise ValueError(f"Span exceeds output length in record {record.get('id')}: {label}")
        if label.start < previous_end:
            raise ValueError(f"Overlapping spans in record {record.get('id')}: {label}")
        if output[label.start : label.end] != label.text:
            raise ValueError(
                f"Span text mismatch in record {record.get('id')}: "
                f"expected {label.text!r}, got {output[label.start:label.end]!r}"
            )
        previous_end = label.end


def ensure_record_dict(record: HallucinationRecord | Mapping[str, Any]) -> dict[str, Any]:
    if isinstance(record, HallucinationRecord):
        return record.to_dict()
    data = dict(record)
    data["labels"] = [coerce_label(label).to_dict() for label in data.get("labels", [])]
    return data


Overwriting tool_hallucination_detection/schema.py


**Explanation:** writes the data-loading module. It loads ToolACE, normalizes tool outputs and final answers, creates split-safe source IDs, and exports RAGTruth-style JSONL files.


In [8]:
%%writefile tool_hallucination_detection/data.py
"""ToolACE loading, normalization, splitting, and JSONL export."""
from __future__ import annotations

import json
import random
from collections import defaultdict
from pathlib import Path
from typing import Any, Iterable, Mapping

from .schema import validate_labels


def load_toolace_rows(split: str = "train", max_records: int | None = None) -> list[dict[str, Any]]:
    """Load raw ToolACE rows from Hugging Face."""
    try:
        from datasets import load_dataset
    except ImportError as exc:
        raise RuntimeError("Install the `datasets` package to load ToolACE.") from exc

    dataset = load_dataset("Team-ACE/ToolACE", split=split)
    if max_records is not None:
        dataset = dataset.select(range(min(max_records, len(dataset))))
    return [dict(row) for row in dataset]


def normalize_toolace_row(row: Mapping[str, Any], row_index: int | str) -> dict[str, Any] | None:
    """Convert one ToolACE row into the project record shape."""
    conversations = list(row.get("conversations") or [])
    if not conversations:
        return None

    user_turn = next((turn for turn in conversations if _turn_role(turn) == "user"), None)
    tool_indices = [i for i, turn in enumerate(conversations) if _turn_role(turn) == "tool"]
    if user_turn is None or not tool_indices:
        return None

    first_tool_index = tool_indices[0]
    tool_call_turn = None
    for turn in reversed(conversations[:first_tool_index]):
        if _turn_role(turn) == "assistant":
            tool_call_turn = turn
            break
    if tool_call_turn is None:
        return None

    final_answer_turn = None
    for turn in conversations[tool_indices[-1] + 1 :]:
        if _turn_role(turn) == "assistant":
            final_answer_turn = turn
            break
    if final_answer_turn is None:
        return None

    output = _clean_text(_turn_value(final_answer_turn))
    if not output:
        return None

    return {
        "id": f"toolace-{row_index}",
        "source_id": f"toolace-{row_index}",
        "split": "unsplit",
        "corruption_type": "base",
        "query": _clean_text(_turn_value(user_turn)),
        "tools": _clean_text(row.get("system", row.get("tools", ""))),
        "tool_call": _clean_text(_turn_value(tool_call_turn)),
        "context": _clean_text(_turn_value(conversations[first_tool_index])),
        "output": output,
        "labels": [],
    }


def normalize_toolace_rows(rows: Iterable[Mapping[str, Any]], max_records: int | None = None) -> list[dict[str, Any]]:
    records: list[dict[str, Any]] = []
    for index, row in enumerate(rows):
        record = normalize_toolace_row(row, index)
        if record is not None:
            records.append(record)
        if max_records is not None and len(records) >= max_records:
            break
    return records


def split_by_source_id(
    records: list[dict[str, Any]],
    seed: int = 42,
    train_ratio: float = 0.70,
    validation_ratio: float = 0.15,
) -> dict[str, list[dict[str, Any]]]:
    """Split records while keeping variants of the same source together."""
    grouped: dict[str, list[dict[str, Any]]] = defaultdict(list)
    for record in records:
        grouped[str(record["source_id"])].append(record)

    source_ids = sorted(grouped)
    rng = random.Random(seed)
    rng.shuffle(source_ids)

    n_total = len(source_ids)
    n_train = int(round(n_total * train_ratio))
    n_validation = int(round(n_total * validation_ratio))
    if n_total >= 3:
        n_validation = max(1, n_validation)
        n_train = min(max(1, n_train), n_total - n_validation - 1)

    train_ids = set(source_ids[:n_train])
    validation_ids = set(source_ids[n_train : n_train + n_validation])

    splits = {"train": [], "validation": [], "test": []}
    for source_id in source_ids:
        split = "train" if source_id in train_ids else "validation" if source_id in validation_ids else "test"
        for record in grouped[source_id]:
            cloned = dict(record)
            cloned["split"] = split
            splits[split].append(cloned)
    return splits


def flatten_splits(splits: Mapping[str, list[dict[str, Any]]]) -> list[dict[str, Any]]:
    return [record for split in ("train", "validation", "test") for record in splits.get(split, [])]


def write_jsonl(records: Iterable[Mapping[str, Any]], path: str | Path) -> Path:
    """Write records with both internal `labels` and assignment-facing `hallucination_labels`.

    The code uses `labels` internally for brevity, while the project PDF names the
    RAGTruth-style field `hallucination_labels`. Keeping both fields makes the
    exported JSONL unambiguous for grading and Hugging Face publication.
    """
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as file:
        for record in records:
            output_record = _with_hallucination_label_alias(record)
            validate_labels(output_record)
            file.write(json.dumps(output_record, ensure_ascii=False) + "\n")
    return path


def read_jsonl(path: str | Path) -> list[dict[str, Any]]:
    with Path(path).open("r", encoding="utf-8") as file:
        return [_with_hallucination_label_alias(json.loads(line)) for line in file if line.strip()]


def _with_hallucination_label_alias(record: Mapping[str, Any]) -> dict[str, Any]:
    data = dict(record)
    labels = list(data.get("labels") or data.get("hallucination_labels") or [])
    data["labels"] = labels
    data["hallucination_labels"] = labels
    return data


def _clean_text(value: Any) -> str:
    return " ".join(str(value).split())


def _turn_role(turn: Mapping[str, Any]) -> str:
    return str(turn.get("from", turn.get("role", turn.get("speaker", "")))).casefold()


def _turn_value(turn: Mapping[str, Any]) -> Any:
    return turn.get("value", turn.get("content", turn.get("text", "")))


def synthetic_toolace_records() -> list[dict[str, Any]]:
    """Small offline sample used by quick smoke runs."""
    return [
        {
            "id": "synthetic-weather",
            "source_id": "synthetic-weather",
            "split": "unsplit",
            "corruption_type": "base",
            "query": "Help me check the weather in Beijing.",
            "tools": "Available tools: Weather_API(location). Calendar_API(date).",
            "tool_call": '[Weather_API(location="Beijing")]',
            "context": '{"location": "Beijing", "weather": "sunny", "temperature": "26 C"}',
            "output": "The weather in Beijing is sunny with a temperature of 26 C.",
            "labels": [],
        },
        {
            "id": "synthetic-market",
            "source_id": "synthetic-market",
            "split": "unsplit",
            "corruption_type": "base",
            "query": "Get the top market trends in the US.",
            "tools": "Available tools: Market Trends API(trend_type, country).",
            "tool_call": '[Market Trends API(trend_type="MARKET_INDEXES", country="us")]',
            "context": '{"country": "US", "trend": "S&P 500 is up", "change": "1.2%"}',
            "output": "The top US market trend is that the S&P 500 is up by 1.2%.",
            "labels": [],
        },
        {
            "id": "synthetic-quotes",
            "source_id": "synthetic-quotes",
            "split": "unsplit",
            "corruption_type": "base",
            "query": "Find me a quote about inspiration.",
            "tools": "Available tools: Quotes by Keywords(word).",
            "tool_call": '[Quotes by Keywords(word="inspiration")]',
            "context": '{"quotes": [{"text": "The only way to do great work is to love what you do.", "author": "Steve Jobs"}]}',
            "output": 'A relevant quote is "The only way to do great work is to love what you do" by Steve Jobs.',
            "labels": [],
        },
    ]


Overwriting tool_hallucination_detection/data.py


**Explanation:** writes the synthetic corruption module. This version is deliberately harder than the earlier template-heavy dataset. It still creates exactly the three hallucination types required by the assignment, but it makes them more diverse: short value substitutions for contradictions, varied unsupported claims for overgeneration, and missing-tool actions selected from tools that are actually unavailable. All injected hallucinations remain span-labeled.


In [9]:
%%writefile tool_hallucination_detection/corruption.py
"""Harder synthetic hallucination injection for tool-calling dialogues.

The assignment requires automatic corruption of ToolACE examples with three hallucination types:
1. contradiction between answer and tool output;
2. overgeneration unsupported by the tool output;
3. missing-tool actions.

This module keeps exact character spans but makes corruptions less template-like than a simple
single appended phrase. The goal is a more useful controlled benchmark, not random noise.
"""
from __future__ import annotations

import hashlib
import re
from typing import Any, Iterable, Mapping

from .schema import SpanLabel, validate_labels

# Word-boundary regexes. These must contain the regex token \b, not the backspace character.
QUOTED_VALUE_RE = re.compile(r"""[\"']([^\"'{}\[\]:,]{2,100})[\"']""")
DATE_RE = re.compile(r"\b(?:19|20)\d{2}[-/]\d{1,2}[-/]\d{1,2}\b|\b\d{1,2}[-/]\d{1,2}[-/](?:19|20)\d{2}\b")
TIME_RE = re.compile(r"\b\d{1,2}:\d{2}\b")
NUMBER_RE = re.compile(r"\b[-+]?\d+(?:\.\d+)?%?\b")
CODE_RE = re.compile(r"\b[A-Z]{2,}[-_]?[A-Z0-9]{2,}\b")
WORD_RE = re.compile(r"\b[A-Za-z][A-Za-z0-9&+.-]{2,}\b")

CONTRADICTION_REPLACEMENTS = {
    "sunny": "rainy", "rainy": "sunny", "cloudy": "clear", "clear": "cloudy",
    "snowy": "hot", "stormy": "calm", "hot": "cold", "cold": "hot",
    "warm": "freezing", "freezing": "warm", "high": "low", "low": "high",
    "up": "down", "down": "up", "increase": "decrease", "decrease": "increase",
    "increased": "decreased", "decreased": "increased", "open": "closed", "closed": "open",
    "available": "unavailable", "unavailable": "available", "active": "inactive", "inactive": "active",
    "enabled": "disabled", "disabled": "enabled", "approved": "rejected", "rejected": "approved",
    "accepted": "denied", "denied": "accepted", "success": "failure", "failure": "success",
    "successful": "failed", "failed": "successful", "true": "false", "false": "true",
    "yes": "no", "no": "yes", "valid": "invalid", "invalid": "valid",
    "confirmed": "cancelled", "cancelled": "confirmed", "completed": "pending", "pending": "completed",
    "paid": "unpaid", "unpaid": "paid", "on": "off", "off": "on",
}

ENTITY_REPLACEMENTS = [
    "London", "Tokyo", "Paris", "Berlin", "New York", "Toronto", "Sydney", "Beijing",
    "Madrid", "Rome", "Singapore", "Dubai", "Boston", "Chicago", "Amsterdam",
]

MISSING_TOOL_ACTIONS = [
    ("flight", " I can also book a flight for you now.", {"book", "flight", "ticket", "reservation"}),
    ("hotel", " I can reserve a hotel room for you as the next step.", {"hotel", "reservation", "booking"}),
    ("payment", " I can proceed with the payment immediately.", {"payment", "pay", "checkout", "purchase"}),
    ("email", " I can send a confirmation email to everyone involved.", {"email", "mail", "send"}),
    ("calendar", " I can add this event to your calendar right away.", {"calendar", "event", "schedule"}),
    ("sms", " I can send an SMS notification to the customer.", {"sms", "text", "message"}),
    ("refund", " I can issue the refund directly from here.", {"refund", "return", "reimburse"}),
    ("call", " I can call the customer and confirm the details.", {"call", "phone", "dial"}),
    ("ride", " I can order a ride to the destination for you.", {"ride", "taxi", "car"}),
    ("restaurant", " I can make a restaurant reservation for tonight.", {"restaurant", "dining", "reservation"}),
    ("cancel", " I can cancel the existing booking for you.", {"cancel", "delete", "void"}),
]

OVERGENERATION_TEMPLATES = [
    ("unsupported_history", " Historical records show that this has been the usual outcome for the past few months."),
    ("unsupported_trend", " The trend is expected to remain stable throughout next week."),
    ("unsupported_cause", " This happened because demand increased sharply in the surrounding area."),
    ("unsupported_user_reaction", " Users were also satisfied with the result according to previous feedback."),
    ("unsupported_risk", " There is no risk of delay or failure based on the available records."),
    ("unsupported_recommendation", " Therefore, the best recommendation is to proceed without any further checks."),
    ("unsupported_forecast", " The same result is very likely to continue for the rest of the month."),
    ("unsupported_external_confirmation", " This was also independently confirmed by an external monitoring system."),
    ("unsupported_comparison", " Compared with similar cases, this is one of the best outcomes recorded recently."),
    ("unsupported_guarantee", " The tool output guarantees that no additional intervention will be required."),
]

FALLBACK_CONTRADICTIONS = [
    ("fallback_status", " However, the returned status should be interpreted as failed rather than successful."),
    ("fallback_value", " However, the answer reports a different value from the one returned by the tool."),
    ("fallback_opposite", " However, this states the opposite of what the tool output supports."),
]

STOP_CANDIDATES = {
    "api", "args", "argument", "arguments", "assistant", "call", "content", "data", "date", "details",
    "false", "function", "id", "input", "json", "message", "name", "none", "null", "object", "output",
    "parameter", "parameters", "query", "request", "response", "result", "results", "return", "returned",
    "schema", "status", "string", "success", "system", "text", "tool", "tools", "true", "type", "user",
    "value", "values", "weather", "location", "city", "country",
}


def build_corrupted_dataset(base_records: Iterable[Mapping[str, Any]], include_clean: bool = True) -> list[dict[str, Any]]:
    """Create clean and three span-labeled hallucinated variants for every usable base record."""
    generated: list[dict[str, Any]] = []
    for base in base_records:
        if include_clean:
            clean = _clone_base(base, "clean")
            validate_labels(clean)
            generated.append(clean)
        for corruption_type, builder in (
            ("tool_contradiction", inject_tool_contradiction),
            ("overgeneration", inject_overgeneration),
            ("missing_tool", inject_missing_tool),
        ):
            corrupted = builder(base)
            if corrupted is None:
                continue
            corrupted["corruption_type"] = corruption_type
            corrupted["id"] = f"{base['source_id']}::{corruption_type}"
            validate_labels(corrupted)
            generated.append(corrupted)
    return generated


def inject_tool_contradiction(record: Mapping[str, Any]) -> dict[str, Any] | None:
    """Substitute a tool-supported answer span with a conflicting value.

    Preference order: dates/times/numbers/codes/quoted values copied from the tool output into the answer.
    These are harder than appending a generic contradiction because the hallucinated span can be short.
    """
    output = str(record["output"])
    context = str(record["context"])
    candidates = _candidate_values(context)

    for candidate in candidates:
        match = _find_case_insensitive(output, candidate)
        if match is None:
            continue
        replacement, style = _replacement_for(candidate, candidates, record)
        if not replacement or replacement.casefold() == candidate.casefold():
            continue
        start, end = match
        new_output = output[:start] + replacement + output[end:]
        label = SpanLabel(
            start=start,
            end=start + len(replacement),
            text=replacement,
            label_type="tool_contradiction",
            meta=f"style={style}; replaced tool-supported value {candidate!r} with {replacement!r}",
        )
        return _variant(record, new_output, [label])

    # Controlled fallback: still span-labeled, but reported as a fallback style in the audit.
    style, addition = _choice(FALLBACK_CONTRADICTIONS, record, "contradiction_fallback")
    start, new_output = _insert_extra(output, addition, record, "contradiction_insert")
    label = SpanLabel(
        start=start,
        end=start + len(addition),
        text=addition,
        label_type="tool_contradiction",
        meta=f"style={style}; fallback contradiction because no copied tool value was found in the answer",
    )
    return _variant(record, new_output, [label])


def inject_overgeneration(record: Mapping[str, Any]) -> dict[str, Any]:
    """Insert plausible but unsupported information absent from the tool output."""
    output = str(record["output"])
    style, addition = _choice(OVERGENERATION_TEMPLATES, record, "overgeneration")
    start, new_output = _insert_extra(output, addition, record, "overgeneration_insert")
    label = SpanLabel(
        start=start,
        end=start + len(addition),
        text=addition,
        label_type="overgeneration",
        meta=f"style={style}; plausible extra claim absent from the tool response",
    )
    return _variant(record, new_output, [label])


def inject_missing_tool(record: Mapping[str, Any]) -> dict[str, Any]:
    """Insert an action that would require a missing tool.

    The chosen action is filtered against available tool text, so we avoid asking an email tool to send email if
    an email-sending tool is actually available.
    """
    output = str(record["output"])
    tools_text = str(record.get("tools", "")).casefold()
    query_text = str(record.get("query", "")).casefold()

    unavailable = []
    for style, sentence, keywords in MISSING_TOOL_ACTIONS:
        if not any(keyword in tools_text for keyword in keywords):
            # prefer actions related to the query, but keep all unavailable actions as fallback choices
            relevance = int(any(keyword in query_text for keyword in keywords))
            unavailable.append((relevance, style, sentence))

    if unavailable:
        unavailable = sorted(unavailable, key=lambda item: (-item[0], item[1]))
        _, style, addition = _choice(unavailable[:4] if len(unavailable) >= 4 else unavailable, record, "missing_tool")
    else:
        style, addition = "generic_external_action", " I can complete the next external action for you now."

    start, new_output = _insert_extra(output, addition, record, "missing_tool_insert")
    label = SpanLabel(
        start=start,
        end=start + len(addition),
        text=addition,
        label_type="missing_tool",
        meta=f"style={style}; action requires a tool not listed as available",
    )
    return _variant(record, new_output, [label])


def _clone_base(record: Mapping[str, Any], corruption_type: str) -> dict[str, Any]:
    return {
        "id": f"{record['source_id']}::{corruption_type}",
        "source_id": str(record["source_id"]),
        "split": str(record.get("split", "unsplit")),
        "corruption_type": corruption_type,
        "query": str(record["query"]),
        "tools": str(record.get("tools", "")),
        "tool_call": str(record.get("tool_call", "")),
        "context": str(record["context"]),
        "output": str(record["output"]),
        "labels": [],
        "hallucination_labels": [],
    }


def _variant(record: Mapping[str, Any], output: str, labels: list[SpanLabel]) -> dict[str, Any]:
    data = _clone_base(record, str(record.get("corruption_type", "corrupted")))
    data["output"] = output
    label_dicts = [label.to_dict() for label in labels]
    data["labels"] = label_dicts
    data["hallucination_labels"] = label_dicts
    return data


def _candidate_values(context: str) -> list[str]:
    """Extract potential facts from tool output in a priority order."""
    candidates: list[str] = []
    seen = set()
    regexes = (DATE_RE, TIME_RE, QUOTED_VALUE_RE, CODE_RE, NUMBER_RE, WORD_RE)
    for regex in regexes:
        for match in regex.finditer(context):
            value = match.group(1) if regex is QUOTED_VALUE_RE else match.group(0)
            value = value.strip().strip(".,;:()[]{}")
            lowered = value.casefold()
            if _useful_candidate(value) and lowered not in seen:
                candidates.append(value)
                seen.add(lowered)
    # Longer and more structured candidates first; this reduces partial replacements.
    return sorted(candidates, key=lambda v: (not _is_structured(v), -len(v)))


def _useful_candidate(value: str) -> bool:
    lowered = value.casefold()
    if len(value) < 2:
        return False
    if lowered in STOP_CANDIDATES:
        return False
    if lowered.startswith(("http", "www")):
        return False
    if len(value) > 100:
        return False
    return True


def _replacement_for(value: str, candidates: list[str] | None, record: Mapping[str, Any]) -> tuple[str, str]:
    lowered = value.casefold()
    if DATE_RE.fullmatch(value):
        return _change_date(value), "date_substitution"
    if TIME_RE.fullmatch(value):
        return _change_time(value), "time_substitution"
    if NUMBER_RE.fullmatch(value):
        return _change_number(value), "number_substitution"
    if lowered in CONTRADICTION_REPLACEMENTS:
        return _match_case(value, CONTRADICTION_REPLACEMENTS[lowered]), "status_or_boolean_flip"
    if CODE_RE.fullmatch(value):
        return _change_code(value), "code_substitution"
    if _looks_like_entity(value):
        return _entity_replacement(value, record), "entity_substitution"

    # Cross-field value swap is used only when the alternative is not identical and not a stopword.
    for other in candidates or []:
        if other.casefold() != lowered and _useful_candidate(other) and not NUMBER_RE.fullmatch(other):
            if abs(len(other) - len(value)) <= max(8, len(value)):
                return _match_case(value, other), "cross_field_value_swap"

    if len(value) <= 6:
        return f"not {value}", "negation_substitution"
    return f"different {value}", "generic_value_substitution"


def _change_number(value: str) -> str:
    has_percent = value.endswith("%")
    numeric_part = value[:-1] if has_percent else value
    try:
        number = float(numeric_part)
    except ValueError:
        return f"not {value}"
    if number == 0:
        changed = 1.0
    else:
        # A noticeable but plausible perturbation.
        sign = 1 if _stable_int(value, "number") % 2 == 0 else -1
        changed = number + sign * max(1.0, abs(number) * 0.15)
    rendered = str(int(round(changed))) if re.fullmatch(r"[-+]?\d+", numeric_part) else f"{changed:.2f}".rstrip("0").rstrip(".")
    return rendered + ("%" if has_percent else "")


def _change_date(value: str) -> str:
    sep = "/" if "/" in value else "-"
    parts = value.split(sep)
    try:
        if len(parts[0]) == 4:
            y, m, d = int(parts[0]), int(parts[1]), int(parts[2])
            d = min(28, d + 3)
            return f"{y:04d}{sep}{m:02d}{sep}{d:02d}"
        d, m, y = int(parts[0]), int(parts[1]), int(parts[2])
        d = min(28, d + 3)
        return f"{d:02d}{sep}{m:02d}{sep}{y:04d}"
    except Exception:
        return value + " later"


def _change_time(value: str) -> str:
    try:
        hour, minute = value.split(":")
        new_hour = (int(hour) + 3) % 24
        return f"{new_hour:02d}:{int(minute):02d}"
    except Exception:
        return "23:59"


def _change_code(value: str) -> str:
    if len(value) < 3:
        return value + "X"
    last = value[-1]
    replacement = "X" if last != "X" else "Y"
    return value[:-1] + replacement


def _entity_replacement(value: str, record: Mapping[str, Any]) -> str:
    pool = [item for item in ENTITY_REPLACEMENTS if item.casefold() != value.casefold()]
    return _match_case(value, _choice(pool, record, f"entity::{value}"))


def _looks_like_entity(value: str) -> bool:
    tokens = value.split()
    if not tokens or len(tokens) > 4:
        return False
    if any(token[:1].isupper() for token in tokens):
        return True
    if value.isupper() and len(value) >= 3:
        return True
    return False


def _is_structured(value: str) -> bool:
    return bool(DATE_RE.fullmatch(value) or TIME_RE.fullmatch(value) or NUMBER_RE.fullmatch(value) or CODE_RE.fullmatch(value))


def _insert_extra(output: str, addition: str, record: Mapping[str, Any], salt: str) -> tuple[int, str]:
    """Insert unsupported text either at the end or after the first sentence."""
    modes = ["append", "append", "after_first_sentence"]
    mode = _choice(modes, record, salt)
    if mode == "after_first_sentence":
        match = re.search(r"[.!?]\s+", output)
        if match is not None and match.end() < len(output):
            start = match.end()
            text = output[:start] + addition + " " + output[start:]
            return start, text
    start = len(output)
    return start, output + addition


def _choice(items: list[Any], record: Mapping[str, Any], salt: str) -> Any:
    key = f"{record.get('source_id', '')}|{record.get('output', '')}|{salt}".encode("utf-8")
    index = int(hashlib.sha256(key).hexdigest()[:8], 16) % len(items)
    return items[index]


def _stable_int(value: str, salt: str) -> int:
    key = f"{value}|{salt}".encode("utf-8")
    return int(hashlib.sha256(key).hexdigest()[:8], 16)


def _match_case(source: str, replacement: str) -> str:
    if source.isupper():
        return replacement.upper()
    if source[:1].isupper():
        return replacement[:1].upper() + replacement[1:]
    return replacement


def _find_case_insensitive(text: str, needle: str) -> tuple[int, int] | None:
    match = re.search(re.escape(needle), text, flags=re.IGNORECASE)
    if match is None:
        return None
    return match.start(), match.end()


Overwriting tool_hallucination_detection/corruption.py


**Explanation:** writes the metric module. It includes sentence-level binary metrics, imbalance-aware metrics, calibration metrics, threshold tuning, and span-level overlap metrics.


In [10]:
%%writefile tool_hallucination_detection/metrics.py
"""Sentence-level and span-level evaluation metrics."""
from __future__ import annotations

from collections import defaultdict
from typing import Any, Mapping, Sequence

import numpy as np

from .schema import coerce_labels, sentence_label


def sentence_metrics(gold: Sequence[int], scores: Sequence[float], threshold: float = 0.5) -> dict[str, float]:
    """Binary sample-level metrics.

    The positive class means: this answer contains at least one hallucinated span.
    In this project the dataset usually contains 1 clean and 3 corrupted variants per
    base dialogue, so positive-class F1 alone can be misleading. We therefore report
    macro-F1, balanced accuracy, PR-AUC, ROC-AUC, and confusion counts as well.
    """
    gold_arr = np.asarray(gold, dtype=int)
    score_arr = np.asarray(scores, dtype=float)
    if gold_arr.size == 0:
        return _empty_sentence_metrics(threshold)

    pred_arr = (score_arr >= threshold).astype(int)

    tp = int(((gold_arr == 1) & (pred_arr == 1)).sum())
    fp = int(((gold_arr == 0) & (pred_arr == 1)).sum())
    fn = int(((gold_arr == 1) & (pred_arr == 0)).sum())
    tn = int(((gold_arr == 0) & (pred_arr == 0)).sum())

    pos_precision = _safe_div(tp, tp + fp)
    pos_recall = _safe_div(tp, tp + fn)
    pos_f1 = _f1(pos_precision, pos_recall)

    neg_precision = _safe_div(tn, tn + fn)
    neg_recall = _safe_div(tn, tn + fp)  # specificity
    neg_f1 = _f1(neg_precision, neg_recall)

    accuracy = _safe_div(tp + tn, len(gold_arr))
    balanced_accuracy = (pos_recall + neg_recall) / 2
    macro_precision = (pos_precision + neg_precision) / 2
    macro_recall = (pos_recall + neg_recall) / 2
    macro_f1 = (pos_f1 + neg_f1) / 2

    metrics = {
        "n": float(len(gold_arr)),
        "positive_rate": float(gold_arr.mean()),
        "predicted_positive_rate": float(pred_arr.mean()),
        "accuracy": accuracy,
        "balanced_accuracy": balanced_accuracy,
        "precision": pos_precision,
        "recall": pos_recall,
        "f1": pos_f1,
        "negative_precision": neg_precision,
        "negative_recall": neg_recall,
        "negative_f1": neg_f1,
        "macro_precision": macro_precision,
        "macro_recall": macro_recall,
        "macro_f1": macro_f1,
        "specificity": neg_recall,
        "threshold": float(threshold),
        "roc_auc": _roc_auc(gold_arr, score_arr),
        "pr_auc": _pr_auc(gold_arr, score_arr),
        "brier": _brier(gold_arr, score_arr),
        "ece_10": _ece(gold_arr, score_arr, n_bins=10),
        "tp": float(tp),
        "fp": float(fp),
        "fn": float(fn),
        "tn": float(tn),
    }
    return metrics


def choose_best_threshold(gold: Sequence[int], scores: Sequence[float], metric: str = "macro_f1") -> float:
    """Pick a validation threshold.

    `macro_f1` is the default because positive-class F1 can reward an
    always-hallucinated classifier on the 1-clean/3-corrupted dataset.
    """
    unique_scores = sorted(set(float(score) for score in scores))
    candidates = {0.5, 1.0}
    for score in unique_scores:
        candidates.add(score + 1e-9)
    for left, right in zip(unique_scores, unique_scores[1:]):
        candidates.add((left + right) / 2)
    if unique_scores:
        candidates.add(min(unique_scores) - 1e-9)
        candidates.add(max(unique_scores) + 1e-9)

    best_threshold = 0.5
    best_value = -1.0
    for threshold in sorted(candidates):
        metrics = sentence_metrics(gold, scores, threshold)
        value = float(metrics.get(metric, metrics.get("f1", 0.0)))
        if value > best_value:
            best_value = value
            best_threshold = threshold
    return float(best_threshold)


def span_metrics(
    gold_spans: Sequence[Sequence[Mapping[str, Any]]],
    predicted_spans: Sequence[Sequence[Mapping[str, Any]]],
    output_lengths: Sequence[int] | None = None,
    iou_threshold: float = 0.5,
) -> dict[str, float]:
    if output_lengths is None:
        max_end = 0
        for spans in list(gold_spans) + list(predicted_spans):
            for span in spans:
                max_end = max(max_end, int(span["end"]))
        output_lengths = [max_end + 1 for _ in gold_spans]

    char_tp = char_fp = char_fn = 0
    exact_tp = exact_fp = exact_fn = 0
    relaxed_tp = relaxed_fp = relaxed_fn = 0

    for gold, pred, length in zip(gold_spans, predicted_spans, output_lengths):
        gold_ranges = [_span_tuple(span) for span in gold]
        pred_ranges = [_span_tuple(span) for span in pred]
        gold_mask = _mask(gold_ranges, length)
        pred_mask = _mask(pred_ranges, length)

        char_tp += int(np.logical_and(gold_mask, pred_mask).sum())
        char_fp += int(np.logical_and(~gold_mask, pred_mask).sum())
        char_fn += int(np.logical_and(gold_mask, ~pred_mask).sum())

        gold_set = set(gold_ranges)
        pred_set = set(pred_ranges)
        exact_tp += len(gold_set & pred_set)
        exact_fp += len(pred_set - gold_set)
        exact_fn += len(gold_set - pred_set)

        matched_gold: set[int] = set()
        matched_pred: set[int] = set()
        for pred_index, pred_span in enumerate(pred_ranges):
            best_gold_index = None
            best_iou = 0.0
            for gold_index, gold_span in enumerate(gold_ranges):
                if gold_index in matched_gold:
                    continue
                score = _iou(pred_span, gold_span)
                if score > best_iou:
                    best_iou = score
                    best_gold_index = gold_index
            if best_gold_index is not None and best_iou >= iou_threshold:
                matched_gold.add(best_gold_index)
                matched_pred.add(pred_index)
        relaxed_tp += len(matched_pred)
        relaxed_fp += len(pred_ranges) - len(matched_pred)
        relaxed_fn += len(gold_ranges) - len(matched_gold)

    char_precision = _safe_div(char_tp, char_tp + char_fp)
    char_recall = _safe_div(char_tp, char_tp + char_fn)
    exact_precision = _safe_div(exact_tp, exact_tp + exact_fp)
    exact_recall = _safe_div(exact_tp, exact_tp + exact_fn)
    relaxed_precision = _safe_div(relaxed_tp, relaxed_tp + relaxed_fp)
    relaxed_recall = _safe_div(relaxed_tp, relaxed_tp + relaxed_fn)

    return {
        "char_precision": char_precision,
        "char_recall": char_recall,
        "char_f1": _f1(char_precision, char_recall),
        "exact_span_precision": exact_precision,
        "exact_span_recall": exact_recall,
        "exact_span_f1": _f1(exact_precision, exact_recall),
        "relaxed_span_precision": relaxed_precision,
        "relaxed_span_recall": relaxed_recall,
        "relaxed_span_f1": _f1(relaxed_precision, relaxed_recall),
    }


def evaluate_predictions(
    records: Sequence[Mapping[str, Any]],
    predictions: Sequence[Mapping[str, Any]],
    threshold: float = 0.5,
) -> dict[str, dict[str, float]]:
    gold = [sentence_label(record) for record in records]
    scores = [float(prediction.get("score", 0.0)) for prediction in predictions]
    gold_spans = [record.get("labels", []) for record in records]
    # Apply the same sentence-level threshold to spans. This keeps the binary and span
    # views consistent for methods that return a score and candidate spans.
    predicted_spans = [
        prediction.get("spans", []) if float(prediction.get("score", 0.0)) >= threshold else []
        for prediction in predictions
    ]
    lengths = [len(str(record.get("output", ""))) for record in records]
    return {
        "sentence": sentence_metrics(gold, scores, threshold),
        "span": span_metrics(gold_spans, predicted_spans, lengths),
    }


def per_type_metrics(
    records: Sequence[Mapping[str, Any]],
    predictions: Sequence[Mapping[str, Any]],
    threshold: float = 0.5,
) -> list[dict[str, float | str]]:
    by_type: dict[str, list[int]] = defaultdict(list)
    for index, record in enumerate(records):
        by_type[str(record.get("corruption_type", "unknown"))].append(index)

    rows: list[dict[str, float | str]] = []
    for corruption_type, indices in sorted(by_type.items()):
        subset_records = [records[index] for index in indices]
        subset_predictions = [predictions[index] for index in indices]
        evaluated = evaluate_predictions(subset_records, subset_predictions, threshold)
        row: dict[str, float | str] = {"corruption_type": corruption_type, "n": len(indices)}
        row.update({f"sentence_{key}": value for key, value in evaluated["sentence"].items()})
        row.update({f"span_{key}": value for key, value in evaluated["span"].items()})
        rows.append(row)
    return rows


def labels_to_char_spans(labels: Sequence[Mapping[str, Any]]) -> list[tuple[int, int]]:
    return [_span_tuple(label) for label in labels]


def _mask(spans: Sequence[tuple[int, int]], length: int) -> np.ndarray:
    mask = np.zeros(max(0, int(length)), dtype=bool)
    for start, end in spans:
        mask[max(0, start) : min(len(mask), end)] = True
    return mask


def _span_tuple(span: Mapping[str, Any]) -> tuple[int, int]:
    return int(span["start"]), int(span["end"])


def _iou(left: tuple[int, int], right: tuple[int, int]) -> float:
    intersection = max(0, min(left[1], right[1]) - max(left[0], right[0]))
    union = max(left[1], right[1]) - min(left[0], right[0])
    return _safe_div(intersection, union)


def _roc_auc(gold: np.ndarray, scores: np.ndarray) -> float:
    if len(set(gold.tolist())) < 2:
        return float("nan")
    try:
        from sklearn.metrics import roc_auc_score
        return float(roc_auc_score(gold, scores))
    except Exception:
        order = np.argsort(scores)
        ranks = np.empty_like(order, dtype=float)
        ranks[order] = np.arange(1, len(scores) + 1)
        pos = gold == 1
        n_pos = int(pos.sum())
        n_neg = int((~pos).sum())
        return float((ranks[pos].sum() - n_pos * (n_pos + 1) / 2) / (n_pos * n_neg))


def _pr_auc(gold: np.ndarray, scores: np.ndarray) -> float:
    if len(set(gold.tolist())) < 2:
        return float("nan")
    try:
        from sklearn.metrics import average_precision_score
        return float(average_precision_score(gold, scores))
    except Exception:
        return float("nan")


def _brier(gold: np.ndarray, scores: np.ndarray) -> float:
    scores = np.clip(scores.astype(float), 0.0, 1.0)
    return float(np.mean((scores - gold.astype(float)) ** 2)) if len(gold) else float("nan")


def _ece(gold: np.ndarray, scores: np.ndarray, n_bins: int = 10) -> float:
    if len(gold) == 0:
        return float("nan")
    scores = np.clip(scores.astype(float), 0.0, 1.0)
    bins = np.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0
    for left, right in zip(bins[:-1], bins[1:]):
        if right == 1.0:
            mask = (scores >= left) & (scores <= right)
        else:
            mask = (scores >= left) & (scores < right)
        if not mask.any():
            continue
        confidence = float(scores[mask].mean())
        accuracy = float(gold[mask].mean())
        ece += float(mask.mean()) * abs(confidence - accuracy)
    return float(ece)


def _safe_div(numerator: float, denominator: float) -> float:
    return float(numerator / denominator) if denominator else 0.0


def _f1(precision: float, recall: float) -> float:
    return _safe_div(2 * precision * recall, precision + recall)


def _empty_sentence_metrics(threshold: float) -> dict[str, float]:
    keys = [
        "n", "positive_rate", "predicted_positive_rate", "accuracy", "balanced_accuracy",
        "precision", "recall", "f1", "negative_precision", "negative_recall", "negative_f1",
        "macro_precision", "macro_recall", "macro_f1", "specificity", "threshold",
        "roc_auc", "pr_auc", "brier", "ece_10", "tp", "fp", "fn", "tn",
    ]
    values = {key: float("nan") for key in keys}
    values["threshold"] = float(threshold)
    values["n"] = 0.0
    return values


def coerce_gold_spans(record: Mapping[str, Any]) -> list[dict[str, Any]]:
    return [label.to_dict() for label in coerce_labels(record.get("labels"))]


Overwriting tool_hallucination_detection/metrics.py


**Explanation:** writes baseline detectors. These include trivial baselines, keyword rules, an NLI verifier, real LettuceDetect, a lightweight LookBackLens-style proxy, and an adapted attention-based LookBackLens baseline. The adapted LookBackLens baseline uses teacher-forced attention maps from a causal LM to measure how much answer-token attention goes back to the tool context.


In [11]:
%%writefile tool_hallucination_detection/baselines.py
"""Baseline detectors for the hallucination benchmark."""
from __future__ import annotations

import hashlib
import math
import re
from dataclasses import dataclass
from typing import Any, Mapping, Sequence

import numpy as np

from .metrics import choose_best_threshold
from .schema import sentence_label

Prediction = dict[str, Any]

SENTENCE_RE = re.compile(r"[^.!?]+[.!?]?")
TOKEN_RE = re.compile(r"[A-Za-z0-9]+")
HIGH_SIGNAL_PATTERNS = [
    re.compile(pattern, re.IGNORECASE)
    for pattern in [
        r"\bpast few months\b",
        r"\bremained stable\b",
        r"\bbook a flight\b",
        r"\breserve a hotel\b",
        r"\bproceed with the payment\b",
        r"\bsend a confirmation email\b",
        r"\badd this to your calendar\b",
        r"\bcomplete the next external action\b",
        r"\bcontradicts the tool output\b",
    ]
]
CONFLICT_PAIRS = [
    ("sunny", "rainy"), ("up", "down"), ("open", "closed"),
    ("available", "unavailable"), ("approved", "rejected"),
    ("success", "failure"), ("true", "false"),
]


def always_clean_predict(records: Sequence[Mapping[str, Any]]) -> list[Prediction]:
    return [{"spans": [], "score": 0.0, "method": "always_clean"} for _ in records]



def always_hallucinated_predict(records: Sequence[Mapping[str, Any]]) -> list[Prediction]:
    """Trivial majority-class baseline: every answer is hallucinated.

    This is important because the generated dataset is usually 75% positive
    (three corrupted variants and one clean variant per source dialogue).
    """
    predictions: list[Prediction] = []
    for record in records:
        output = str(record.get("output", ""))
        spans = [_full_output_span(output, 1.0, "always_hallucinated")] if output else []
        predictions.append({"spans": spans, "score": 1.0, "method": "always_hallucinated"})
    return predictions


def random_balanced_predict(records: Sequence[Mapping[str, Any]], seed: int = 42) -> list[Prediction]:
    """Deterministic random baseline with uniform scores."""
    predictions: list[Prediction] = []
    for record in records:
        score = _stable_random_score(record, seed=seed)
        output = str(record.get("output", ""))
        spans = [_full_output_span(output, score, "random_balanced")] if output and score >= 0.5 else []
        predictions.append({"spans": spans, "score": score, "method": "random_balanced"})
    return predictions


def keyword_tool_action_predict(records: Sequence[Mapping[str, Any]]) -> list[Prediction]:
    """Fast sanity baseline for injected hallucination types."""
    return [_keyword_prediction(record) for record in records]


def nli_sentence_verifier_predict(
    records: Sequence[Mapping[str, Any]],
    model_name: str = "MoritzLaurer/deberta-v3-base-mnli-fever-anli",
    fallback_to_keyword: bool = True,
) -> list[Prediction]:
    """Optional open-source NLI baseline."""
    try:
        from transformers import pipeline
    except Exception:
        if fallback_to_keyword:
            return keyword_tool_action_predict(records)
        raise

    try:
        classifier = pipeline(
            "text-classification",
            model=model_name,
            top_k=None,
            truncation=True,
            device=0 if _cuda_available() else -1,
        )
    except Exception:
        if fallback_to_keyword:
            return keyword_tool_action_predict(records)
        raise

    predictions: list[Prediction] = []
    for record in records:
        spans = []
        max_score = 0.0
        context = f"Question: {record['query']}\nTool output: {record['context']}"
        for start, end, sentence in _iter_sentence_spans(str(record["output"])):
            if not sentence.strip():
                continue
            result = classifier({"text": context, "text_pair": sentence})
            label_scores = _normalize_pipeline_scores(result)
            contradiction = max([score for label, score in label_scores.items() if "contradiction" in label], default=0.0)
            neutral = max([score for label, score in label_scores.items() if "neutral" in label], default=0.0)
            score = max(contradiction, neutral * 0.65)
            if score >= 0.5:
                spans.append(_span(start, end, sentence, score, "nli_sentence_verifier"))
            max_score = max(max_score, score)
        predictions.append({"spans": spans, "score": max_score, "method": "nli_sentence_verifier"})
    return predictions


def lettucedetect_predict(
    records: Sequence[Mapping[str, Any]],
    model_path: str | None = None,
    fallback_to_keyword: bool = False,
) -> list[Prediction]:
    """Run the real LettuceDetect baseline.

    When ``fallback_to_keyword`` is False, loading/import errors are raised so the
    final table cannot silently report a keyword method as LettuceDetect.
    """
    try:
        from lettucedetect.models.inference import HallucinationDetector
    except Exception:
        if fallback_to_keyword:
            preds = keyword_tool_action_predict(records)
            for pred in preds:
                pred["method"] = "lettucedetect_fallback_keyword"
            return preds
        raise

    try:
        detector = HallucinationDetector(method="transformer", model_path=model_path) if model_path else HallucinationDetector(method="transformer")
    except Exception:
        if fallback_to_keyword:
            preds = keyword_tool_action_predict(records)
            for pred in preds:
                pred["method"] = "lettucedetect_fallback_keyword"
            return preds
        raise

    predictions: list[Prediction] = []
    for record in records:
        raw = detector.predict(
            context=str(record["context"]),
            question=str(record["query"]),
            answer=str(record["output"]),
            output_format="spans",
        )
        spans = []
        for item in raw:
            start = int(_get_field(item, "start"))
            end = int(_get_field(item, "end"))
            confidence = float(_get_field(item, "hallucination_score", _get_field(item, "confidence", 0.5)))
            text = str(record["output"])[start:end]
            spans.append(_span(start, end, text, confidence, "lettucedetect_real"))
        score = max([span["confidence"] for span in spans], default=0.0)
        predictions.append({"spans": spans, "score": score, "method": "lettucedetect_real"})
    return predictions


@dataclass
class RandomPriorBaseline:
    """Random baseline whose expected positive rate is fitted on validation data."""
    prior: float = 0.5
    seed: int = 42

    def fit(self, records: Sequence[Mapping[str, Any]]) -> "RandomPriorBaseline":
        if records:
            self.prior = float(np.mean([sentence_label(record) for record in records]))
        return self

    def predict(self, records: Sequence[Mapping[str, Any]]) -> list[Prediction]:
        predictions: list[Prediction] = []
        # A uniform score can be thresholded on validation; the default span decision
        # uses the fitted prior only for an interpretable random positive rate.
        cutoff = 1.0 - float(self.prior)
        for record in records:
            score = _stable_random_score(record, seed=self.seed + 17)
            output = str(record.get("output", ""))
            spans = [_full_output_span(output, score, "random_prior")] if output and score >= cutoff else []
            predictions.append({"spans": spans, "score": score, "method": "random_prior"})
        return predictions


@dataclass
class LookBackLensStyleBaseline:
    """Lightweight lexical proxy kept only as an extra sanity baseline.

    This class does not use attention maps, so it is not reported as the main
    LookBackLens baseline in the final method discussion. The attention-based
    adaptation is implemented by AttentionLookBackLensBaseline below.
    """
    threshold: float = 0.5
    weights: np.ndarray | None = None

    def fit(self, records: Sequence[Mapping[str, Any]]) -> "LookBackLensStyleBaseline":
        features = []
        labels = []
        for record in records:
            record_labels = record.get("labels", [])
            for start, end, sentence in _iter_sentence_spans(str(record["output"])):
                features.append(_support_features(record, sentence))
                labels.append(int(_overlaps_any(start, end, record_labels)))

        if len(set(labels)) < 2:
            self.weights = np.zeros(5)
            self.threshold = 0.5
            return self

        try:
            from sklearn.linear_model import LogisticRegression
            clf = LogisticRegression(max_iter=1000, class_weight="balanced")
            clf.fit(np.asarray(features), np.asarray(labels))
            scores = clf.predict_proba(np.asarray(features))[:, 1]
            self.threshold = choose_best_threshold(labels, scores)
            self.weights = np.concatenate([clf.intercept_, clf.coef_.ravel()])
        except Exception:
            self.weights = np.asarray([-1.0, -2.0, 1.2, 0.8, 0.6])
            scores = [_sigmoid(float(np.dot(self.weights, [1.0, *feat]))) for feat in features]
            self.threshold = choose_best_threshold(labels, scores)
        return self

    def predict(self, records: Sequence[Mapping[str, Any]]) -> list[Prediction]:
        predictions: list[Prediction] = []
        for record in records:
            spans = []
            max_score = 0.0
            for start, end, sentence in _iter_sentence_spans(str(record["output"])):
                features = _support_features(record, sentence)
                score = self._score(features)
                if score >= self.threshold:
                    spans.append(_span(start, end, sentence, score, "lookback_lens_style_proxy"))
                max_score = max(max_score, score)
            predictions.append({"spans": spans, "score": max_score, "method": "lookback_lens_style_proxy"})
        return predictions

    def _score(self, features: Sequence[float]) -> float:
        if self.weights is None:
            support, novelty, action, conflict = features
            return max(0.0, min(1.0, 0.15 + 0.55 * novelty + 0.25 * action + 0.20 * conflict - 0.25 * support))
        return _sigmoid(float(np.dot(self.weights, [1.0, *features])))


@dataclass
class AttentionLookBackLensBaseline:
    """Adapted attention-based LookBackLens baseline for fixed answers.

    Original LookBackLens uses attention traces during autoregressive generation.
    Our benchmark already contains fixed clean/corrupted answers, so this class uses
    teacher forcing: it feeds ``Question + Context + Answer`` into a small causal LM
    and extracts attention from answer tokens back to context tokens. A logistic
    classifier is then trained on these attention features.
    """
    model_name: str = "distilgpt2"
    max_length: int = 384
    max_answer_tokens: int = 96
    max_train_records: int = 240
    seed: int = 42
    classifier: Any | None = None
    threshold: float = 0.5
    _tokenizer: Any | None = None
    _model: Any | None = None

    def fit(self, records: Sequence[Mapping[str, Any]]) -> "AttentionLookBackLensBaseline":
        train_records = list(records)
        if self.max_train_records and len(train_records) > self.max_train_records:
            # Deterministic subsample to keep the baseline feasible in Colab.
            scored = [(_stable_random_score(record, seed=self.seed + 101), record) for record in train_records]
            train_records = [record for _, record in sorted(scored, key=lambda item: item[0])[: self.max_train_records]]

        features = [self._features(record) for record in train_records]
        labels = [sentence_label(record) for record in train_records]
        if len(set(labels)) < 2:
            self.classifier = None
            self.threshold = 0.5
            return self

        try:
            from sklearn.linear_model import LogisticRegression
            clf = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=self.seed)
            clf.fit(np.asarray(features, dtype=float), np.asarray(labels, dtype=int))
            scores = clf.predict_proba(np.asarray(features, dtype=float))[:, 1]
            self.classifier = clf
            self.threshold = choose_best_threshold(labels, scores, metric="macro_f1")
        except Exception:
            self.classifier = None
            scores = [self._fallback_score(feat) for feat in features]
            self.threshold = choose_best_threshold(labels, scores, metric="macro_f1")
        return self

    def predict(self, records: Sequence[Mapping[str, Any]]) -> list[Prediction]:
        predictions: list[Prediction] = []
        for record in records:
            features = self._features(record)
            score = self._score_features(features)
            output = str(record.get("output", ""))
            spans = [_full_output_span(output, score, "attention_lookback_lens_adapted")] if output and score >= self.threshold else []
            predictions.append({"spans": spans, "score": score, "method": "attention_lookback_lens_adapted"})
        return predictions

    def _score_features(self, features: Sequence[float]) -> float:
        arr = np.asarray([features], dtype=float)
        if self.classifier is not None:
            try:
                return float(self.classifier.predict_proba(arr)[0, 1])
            except Exception:
                pass
        return self._fallback_score(features)

    def _fallback_score(self, features: Sequence[float]) -> float:
        # features: context_attention, answer_attention, lookback_ratio, lexical_support, novelty, length_norm
        context_attention, answer_attention, lookback_ratio, lexical_support, novelty, length_norm = features
        risk = 1.2 * (1.0 - lookback_ratio) + 0.7 * novelty + 0.2 * length_norm - 0.4 * lexical_support
        return float(max(0.0, min(1.0, risk / 2.0)))

    def _features(self, record: Mapping[str, Any]) -> list[float]:
        try:
            return self._attention_features(record)
        except Exception:
            # If attention extraction fails for one example, fall back to transparent
            # lexical support features so the whole run does not become unusable.
            support, novelty, action, conflict = _support_features(record, str(record.get("output", "")))
            return [support, 1.0 - support, support, support, novelty, min(1.0, len(_tokens(str(record.get("output", "")))) / 80.0)]

    def _ensure_model(self) -> None:
        if self._tokenizer is not None and self._model is not None:
            return
        import torch
        from transformers import AutoModelForCausalLM, AutoTokenizer
        self._tokenizer = AutoTokenizer.from_pretrained(self.model_name, use_fast=True)
        if getattr(self._tokenizer, "pad_token", None) is None:
            self._tokenizer.pad_token = self._tokenizer.eos_token
        try:
            self._model = AutoModelForCausalLM.from_pretrained(
                self.model_name,
                output_attentions=True,
                attn_implementation="eager",
            )
        except TypeError:
            self._model = AutoModelForCausalLM.from_pretrained(self.model_name, output_attentions=True)
        device = "cuda" if torch.cuda.is_available() else "cpu"
        self._model.to(device)
        self._model.eval()

    def _attention_features(self, record: Mapping[str, Any]) -> list[float]:
        import torch
        self._ensure_model()
        tokenizer = self._tokenizer
        model = self._model
        assert tokenizer is not None and model is not None

        question = str(record.get("query", ""))
        context = str(record.get("context", ""))
        answer = str(record.get("output", ""))
        prefix_before_context = f"Question: {question}\nContext: "
        middle = "\nAnswer: "
        full_text = prefix_before_context + context + middle + answer
        context_start = len(prefix_before_context)
        context_end = context_start + len(context)
        answer_start = context_end + len(middle)
        answer_end = answer_start + len(answer)

        encoded = tokenizer(
            full_text,
            return_offsets_mapping=True,
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt",
        )
        offsets = encoded.pop("offset_mapping")[0].tolist()
        device = next(model.parameters()).device
        encoded = {key: value.to(device) for key, value in encoded.items()}

        context_indices = [i for i, (s, e) in enumerate(offsets) if e > context_start and s < context_end and e > s]
        answer_indices = [i for i, (s, e) in enumerate(offsets) if e > answer_start and s < answer_end and e > s]
        if self.max_answer_tokens and len(answer_indices) > self.max_answer_tokens:
            answer_indices = answer_indices[-self.max_answer_tokens :]
        if not context_indices or not answer_indices:
            support, novelty, _action, _conflict = _support_features(record, answer)
            return [support, 1.0 - support, support, support, novelty, min(1.0, len(_tokens(answer)) / 80.0)]

        with torch.no_grad():
            outputs = model(**encoded, output_attentions=True, use_cache=False)
        attentions = outputs.attentions
        selected_layers = attentions[-4:] if len(attentions) >= 4 else attentions

        context_mass_values = []
        prev_answer_mass_values = []
        ratio_values = []
        context_tensor = torch.tensor(context_indices, device=device, dtype=torch.long)
        answer_set = set(answer_indices)
        for attn in selected_layers:
            # attn shape: [batch, heads, seq, seq]
            attn0 = attn[0].float()
            for pos in answer_indices:
                if pos <= 0:
                    continue
                head_rows = attn0[:, pos, :]
                context_mass = head_rows.index_select(-1, context_tensor).sum(dim=-1).mean().item()
                prev_answer_indices = [idx for idx in answer_indices if idx < pos and idx in answer_set]
                if prev_answer_indices:
                    prev_tensor = torch.tensor(prev_answer_indices, device=device, dtype=torch.long)
                    prev_answer_mass = head_rows.index_select(-1, prev_tensor).sum(dim=-1).mean().item()
                else:
                    prev_answer_mass = 0.0
                denom = context_mass + prev_answer_mass + 1e-8
                ratio = context_mass / denom
                context_mass_values.append(context_mass)
                prev_answer_mass_values.append(prev_answer_mass)
                ratio_values.append(ratio)

        support, novelty, _action, _conflict = _support_features(record, answer)
        length_norm = min(1.0, len(_tokens(answer)) / 80.0)
        return [
            float(np.mean(context_mass_values)) if context_mass_values else 0.0,
            float(np.mean(prev_answer_mass_values)) if prev_answer_mass_values else 0.0,
            float(np.mean(ratio_values)) if ratio_values else 0.0,
            float(support),
            float(novelty),
            float(length_norm),
        ]


@dataclass
class ImprovedEnsembleDetector:
    threshold: float = 0.5
    lookback: LookBackLensStyleBaseline | None = None

    def fit(self, validation_records: Sequence[Mapping[str, Any]]) -> "ImprovedEnsembleDetector":
        self.lookback = LookBackLensStyleBaseline().fit(validation_records)
        scores = [self._score_record(record) for record in validation_records]
        gold = [sentence_label(record) for record in validation_records]
        self.threshold = choose_best_threshold(gold, scores)
        return self

    def predict(self, records: Sequence[Mapping[str, Any]]) -> list[Prediction]:
        predictions = []
        for record in records:
            keyword = _keyword_prediction(record)
            lookback_prediction = (self.lookback or LookBackLensStyleBaseline()).predict([record])[0]
            score = self._score_record(record, keyword, lookback_prediction)
            spans = _merge_spans(keyword["spans"] or lookback_prediction["spans"], str(record["output"]))
            if score < self.threshold:
                spans = []
            predictions.append({"spans": spans, "score": score, "method": "improved_ensemble"})
        return predictions

    def _score_record(
        self,
        record: Mapping[str, Any],
        keyword: Prediction | None = None,
        lookback_prediction: Prediction | None = None,
    ) -> float:
        keyword = keyword or _keyword_prediction(record)
        lookback_prediction = lookback_prediction or (self.lookback or LookBackLensStyleBaseline()).predict([record])[0]
        tool_rule_score = _tool_rule_score(record)
        return float(min(
            1.0,
            0.65 * float(lookback_prediction.get("score", 0.0))
            + 0.20 * float(keyword.get("score", 0.0))
            + 0.15 * tool_rule_score,
        ))


def _full_output_span(output: str, confidence: float, method: str) -> dict[str, Any]:
    return {
        "start": 0,
        "end": len(output),
        "text": output,
        "confidence": float(confidence),
        "label_type": method,
    }


def _stable_random_score(record: Mapping[str, Any], seed: int = 42) -> float:
    key = f"{seed}|{record.get('id', '')}|{record.get('source_id', '')}|{record.get('output', '')}".encode("utf-8")
    digest = hashlib.sha256(key).hexdigest()[:16]
    return int(digest, 16) / float(16 ** 16 - 1)


def _keyword_prediction(record: Mapping[str, Any]) -> Prediction:
    output = str(record["output"])
    spans = []
    max_score = 0.0
    for start, end, sentence in _iter_sentence_spans(output):
        score = _sentence_heuristic_score(record, sentence)
        if score >= 0.5:
            spans.append(_span(start, end, sentence, score, "keyword_tool_action"))
        max_score = max(max_score, score)
    return {"spans": _merge_spans(spans, output), "score": max_score, "method": "keyword_tool_action"}


def _sentence_heuristic_score(record: Mapping[str, Any], sentence: str) -> float:
    score = 0.0
    if any(pattern.search(sentence) for pattern in HIGH_SIGNAL_PATTERNS):
        score = max(score, 0.92)
    support, novelty, action, conflict = _support_features(record, sentence)
    if action:
        score = max(score, 0.75)
    if conflict:
        score = max(score, 0.70)
    if novelty > 0.80 and len(_tokens(sentence)) >= 8:
        score = max(score, 0.55)
    return score


def _support_features(record: Mapping[str, Any], sentence: str) -> list[float]:
    context_tokens = set(_tokens(str(record.get("context", ""))) + _tokens(str(record.get("query", ""))))
    sentence_tokens = _tokens(sentence)
    if not sentence_tokens:
        return [1.0, 0.0, 0.0, 0.0]
    overlap = sum(1 for token in sentence_tokens if token in context_tokens)
    support = overlap / len(sentence_tokens)
    novelty = 1.0 - support
    action = float(_has_missing_tool_action(sentence, str(record.get("tools", ""))))
    conflict = float(_has_known_conflict(sentence, str(record.get("context", ""))))
    return [support, novelty, action, conflict]


def _tool_rule_score(record: Mapping[str, Any]) -> float:
    output = str(record.get("output", ""))
    tools = str(record.get("tools", ""))
    if any(_has_missing_tool_action(sentence, tools) for _, _, sentence in _iter_sentence_spans(output)):
        return 0.9
    if _has_known_conflict(output, str(record.get("context", ""))):
        return 0.7
    return 0.0


def _iter_sentence_spans(text: str) -> list[tuple[int, int, str]]:
    spans = []
    for match in SENTENCE_RE.finditer(text):
        sentence = match.group(0)
        if sentence.strip():
            spans.append((match.start(), match.end(), sentence))
    return spans or [(0, len(text), text)]


def _tokens(text: str) -> list[str]:
    return [token.casefold() for token in TOKEN_RE.findall(text)]


def _has_missing_tool_action(sentence: str, tools: str) -> bool:
    lowered_sentence = sentence.casefold()
    lowered_tools = tools.casefold()
    for hint, pattern in [
        ("flight", "book a flight"),
        ("hotel", "reserve a hotel"),
        ("payment", "payment"),
        ("email", "confirmation email"),
        ("calendar", "calendar"),
    ]:
        if pattern in lowered_sentence and hint not in lowered_tools:
            return True
    return "external action" in lowered_sentence


def _has_known_conflict(answer: str, context: str) -> bool:
    answer_lower = answer.casefold()
    context_lower = context.casefold()
    for left, right in CONFLICT_PAIRS:
        if left in context_lower and right in answer_lower:
            return True
        if right in context_lower and left in answer_lower:
            return True
    return False


def _overlaps_any(start: int, end: int, labels: Sequence[Mapping[str, Any]]) -> bool:
    for label in labels:
        if max(start, int(label["start"])) < min(end, int(label["end"])):
            return True
    return False


def _span(start: int, end: int, text: str, confidence: float, method: str) -> dict[str, Any]:
    return {"start": int(start), "end": int(end), "text": text, "confidence": float(confidence), "label_type": method}


def _normalize_pipeline_scores(result: Any) -> dict[str, float]:
    items: list[Mapping[str, Any]] = []
    def collect(value: Any) -> None:
        if isinstance(value, Mapping):
            if "label" in value and "score" in value:
                items.append(value)
            return
        if isinstance(value, list):
            for nested in value:
                collect(nested)
    collect(result)
    return {str(item["label"]).lower(): float(item["score"]) for item in items}


def _get_field(item: Any, name: str, default: Any = None) -> Any:
    if isinstance(item, Mapping):
        return item.get(name, default)
    return getattr(item, name, default)


def _merge_spans(spans: Sequence[Mapping[str, Any]], output: str) -> list[dict[str, Any]]:
    if not spans:
        return []
    ordered = sorted(spans, key=lambda span: (int(span["start"]), int(span["end"])))
    merged: list[dict[str, Any]] = []
    for span in ordered:
        start, end = int(span["start"]), int(span["end"])
        confidence = float(span.get("confidence", 0.5))
        if not merged or start > int(merged[-1]["end"]):
            merged.append(_span(start, end, output[start:end], confidence, str(span.get("label_type", "merged"))))
        else:
            merged[-1]["end"] = max(int(merged[-1]["end"]), end)
            merged[-1]["text"] = output[int(merged[-1]["start"]) : int(merged[-1]["end"])]
            merged[-1]["confidence"] = max(float(merged[-1].get("confidence", 0.0)), confidence)
    return merged


def _sigmoid(value: float) -> float:
    return 1.0 / (1.0 + math.exp(-value))


def _cuda_available() -> bool:
    try:
        import torch
        return bool(torch.cuda.is_available())
    except Exception:
        return False


Overwriting tool_hallucination_detection/baselines.py


**Explanation:** writes the ModernBERT token-classification model. The implementation masks question/context tokens from the loss, removes `offset_mapping` before collation, uses class-weighted token loss to address rare hallucinated tokens, and supports separate sentence/span threshold tuning on validation.


In [12]:
%%writefile tool_hallucination_detection/modeling.py
"""Optional transformer token-classification model for span hallucination detection."""
from __future__ import annotations

import inspect
from pathlib import Path
from typing import Any, Mapping, Sequence

import numpy as np

DEFAULT_MODEL_NAME = "answerdotai/ModernBERT-base"


def format_model_input(record: Mapping[str, Any]) -> tuple[str, int]:
    prefix = f"Question: {record['query']}\nContext: {record['context']}\nAnswer: "
    return prefix + str(record["output"]), len(prefix)


def tokenize_with_span_labels(record: Mapping[str, Any], tokenizer: Any, max_length: int = 2048) -> dict[str, Any]:
    """Tokenize full input and create token labels over answer characters.

    Important implementation details:
    - Query/context tokens are ignored with label -100.
    - Only answer tokens can contribute to the loss.
    - `offset_mapping` is popped before collation. Keeping it in the dataset
      causes DataCollatorForTokenClassification to fail on variable nested lists.
    """
    full_text, answer_start = format_model_input(record)
    encoded = tokenizer(
        full_text,
        truncation=True,
        max_length=max_length,
        return_offsets_mapping=True,
    )

    offset_mapping = encoded.pop("offset_mapping")

    gold_spans = [
        (answer_start + int(label["start"]), answer_start + int(label["end"]))
        for label in record.get("labels", [])
    ]

    labels = []
    for start, end in offset_mapping:
        if end <= answer_start or start == end:
            labels.append(-100)
        elif any(max(start, gold_start) < min(end, gold_end) for gold_start, gold_end in gold_spans):
            labels.append(1)
        else:
            labels.append(0)

    encoded["labels"] = labels
    return encoded


class WeightedTokenClassificationTrainerMixin:
    """Trainer mixin with class-weighted token-level cross entropy.

    Hallucinated tokens are much rarer than normal answer tokens. Weighted loss
    prevents the classifier from minimizing loss by mostly predicting the
    negative class. The ignore index -100 masks question/context tokens.
    """

    def __init__(self, *args: Any, class_weights: Any = None, **kwargs: Any) -> None:
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model: Any, inputs: dict[str, Any], return_outputs: bool = False, num_items_in_batch: Any = None):
        import torch
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        weights = self.class_weights.to(logits.device) if self.class_weights is not None else None
        loss_fct = torch.nn.CrossEntropyLoss(weight=weights, ignore_index=-100)
        loss = loss_fct(logits.view(-1, logits.shape[-1]), labels.view(-1))
        return (loss, outputs) if return_outputs else loss


def train_token_classifier(
    train_records: Sequence[Mapping[str, Any]],
    validation_records: Sequence[Mapping[str, Any]],
    output_dir: str | Path = "artifacts/modernbert-token-classifier",
    model_name: str = DEFAULT_MODEL_NAME,
    epochs: float = 1.0,
    batch_size: int = 2,
    max_length: int = 2048,
    learning_rate: float = 2e-5,
    use_class_weights: bool = True,
    positive_class_weight_cap: float = 12.0,
) -> Path:
    """Fine-tune a token classifier on answer spans."""
    import torch
    from datasets import Dataset
    from transformers import (
        AutoModelForTokenClassification,
        AutoTokenizer,
        DataCollatorForTokenClassification,
        Trainer,
        TrainingArguments,
    )

    class WeightedTrainer(WeightedTokenClassificationTrainerMixin, Trainer):
        pass

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    train_dataset = Dataset.from_list(list(train_records)).map(
        lambda record: tokenize_with_span_labels(record, tokenizer, max_length=max_length),
        remove_columns=list(train_records[0].keys()),
    )
    validation_dataset = Dataset.from_list(list(validation_records)).map(
        lambda record: tokenize_with_span_labels(record, tokenizer, max_length=max_length),
        remove_columns=list(train_records[0].keys()),
    )

    model = AutoModelForTokenClassification.from_pretrained(model_name, num_labels=2)
    output_dir = Path(output_dir)
    args = TrainingArguments(**_training_args_kwargs(
        TrainingArguments=TrainingArguments,
        output_dir=output_dir,
        batch_size=batch_size,
        epochs=epochs,
        learning_rate=learning_rate,
    ))

    base_trainer_kwargs = _trainer_kwargs(
        Trainer=Trainer,
        model=model,
        args=args,
        train_dataset=train_dataset,
        eval_dataset=validation_dataset,
        tokenizer=tokenizer,
        data_collator=DataCollatorForTokenClassification(tokenizer),
    )

    if use_class_weights:
        weights = _estimate_class_weights(train_dataset, positive_cap=positive_class_weight_cap)
        class_weights = torch.tensor(weights, dtype=torch.float32)
        trainer = WeightedTrainer(class_weights=class_weights, **base_trainer_kwargs)
    else:
        trainer = Trainer(**base_trainer_kwargs)

    trainer.train()
    trainer.save_model(str(output_dir))
    tokenizer.save_pretrained(str(output_dir))
    return output_dir


def _estimate_class_weights(dataset: Any, positive_cap: float = 12.0) -> list[float]:
    counts = {0: 0, 1: 0}
    for row in dataset:
        for label in row.get("labels", []):
            if int(label) in counts:
                counts[int(label)] += 1
    neg = max(1, counts[0])
    pos = max(1, counts[1])
    total = neg + pos
    weight_neg = total / (2.0 * neg)
    weight_pos = min(total / (2.0 * pos), float(positive_cap))
    # Normalize so the negative class stays close to 1.0 and only the positive
    # upweighting changes substantially.
    if weight_neg > 0:
        weight_pos = weight_pos / weight_neg
        weight_neg = 1.0
    return [float(weight_neg), float(weight_pos)]


def _training_args_kwargs(TrainingArguments: Any, output_dir: Path, batch_size: int, epochs: float, learning_rate: float) -> dict[str, Any]:
    """Build TrainingArguments kwargs across transformers versions."""
    import torch
    signature = inspect.signature(TrainingArguments.__init__)
    kwargs: dict[str, Any] = {
        "output_dir": str(output_dir),
        "per_device_train_batch_size": batch_size,
        "per_device_eval_batch_size": batch_size,
        "num_train_epochs": epochs,
        "learning_rate": learning_rate,
        "weight_decay": 0.01,
        "save_strategy": "epoch",
        "logging_steps": 20,
        "load_best_model_at_end": True,
        "metric_for_best_model": "eval_loss",
        "report_to": [],
        "remove_unused_columns": False,
        "fp16": bool(torch.cuda.is_available()),
        "gradient_accumulation_steps": 1,
        "seed": 42,
        "data_seed": 42,
    }
    if "eval_strategy" in signature.parameters:
        kwargs["eval_strategy"] = "epoch"
    else:
        kwargs["evaluation_strategy"] = "epoch"
    return {key: value for key, value in kwargs.items() if key in signature.parameters}


def _trainer_kwargs(Trainer: Any, model: Any, args: Any, train_dataset: Any, eval_dataset: Any, tokenizer: Any, data_collator: Any) -> dict[str, Any]:
    """Build Trainer kwargs across transformers versions."""
    signature = inspect.signature(Trainer.__init__)
    kwargs: dict[str, Any] = {
        "model": model,
        "args": args,
        "train_dataset": train_dataset,
        "eval_dataset": eval_dataset,
        "data_collator": data_collator,
    }
    if "processing_class" in signature.parameters:
        kwargs["processing_class"] = tokenizer
    elif "tokenizer" in signature.parameters:
        kwargs["tokenizer"] = tokenizer
    return kwargs


def predict_with_token_classifier(
    records: Sequence[Mapping[str, Any]],
    model_path: str | Path,
    threshold: float = 0.5,
    max_length: int = 2048,
) -> list[dict[str, Any]]:
    """Predict hallucinated answer spans with a fine-tuned token classifier."""
    import torch
    from transformers import AutoModelForTokenClassification, AutoTokenizer

    tokenizer = AutoTokenizer.from_pretrained(str(model_path))
    model = AutoModelForTokenClassification.from_pretrained(str(model_path))
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model.to(device).eval()

    predictions = []
    for record in records:
        full_text, answer_start = format_model_input(record)
        encoded = tokenizer(
            full_text,
            truncation=True,
            max_length=max_length,
            return_offsets_mapping=True,
            return_tensors="pt",
        )
        offsets = encoded.pop("offset_mapping")[0].tolist()
        encoded = {key: value.to(device) for key, value in encoded.items()}
        with torch.inference_mode():
            logits = model(**encoded).logits[0]
            probs = torch.softmax(logits, dim=-1)[:, 1].detach().cpu().numpy()

        token_spans = []
        answer_probabilities = []
        for (start, end), probability in zip(offsets, probs):
            if end <= answer_start or start == end:
                continue
            probability = float(probability)
            answer_probabilities.append(probability)
            if probability >= threshold:
                token_spans.append((start - answer_start, end - answer_start, probability))
        spans = _aggregate_token_spans(token_spans, str(record["output"]))
        predictions.append({
            "spans": spans,
            "score": max(answer_probabilities, default=0.0),
            "method": "modernbert_token_classifier",
        })
    return predictions


def tune_token_classifier_thresholds(
    validation_records: Sequence[Mapping[str, Any]],
    model_path: str | Path,
    max_length: int = 2048,
    sentence_metric: str = "macro_f1",
    span_metric: str = "char_f1",
    span_threshold_grid: Sequence[float] = tuple(np.round(np.linspace(0.10, 0.90, 9), 2)),
) -> dict[str, float]:
    """Tune sentence and span thresholds separately on validation data."""
    from .metrics import choose_best_threshold, evaluate_predictions
    from .schema import sentence_label

    score_predictions = predict_with_token_classifier(validation_records, model_path, threshold=0.5, max_length=max_length)
    gold = [sentence_label(record) for record in validation_records]
    scores = [float(prediction.get("score", 0.0)) for prediction in score_predictions]
    sentence_threshold = choose_best_threshold(gold, scores, metric=sentence_metric)

    best_span_threshold = 0.5
    best_span_value = -1.0
    for threshold in span_threshold_grid:
        predictions = predict_with_token_classifier(validation_records, model_path, threshold=float(threshold), max_length=max_length)
        metrics = evaluate_predictions(validation_records, predictions, threshold=sentence_threshold)["span"]
        value = float(metrics.get(span_metric, 0.0))
        if value > best_span_value:
            best_span_value = value
            best_span_threshold = float(threshold)

    return {
        "sentence_threshold": float(sentence_threshold),
        "span_threshold": float(best_span_threshold),
        f"validation_{span_metric}": float(best_span_value),
    }


def _aggregate_token_spans(token_spans: Sequence[tuple[int, int, float]], output: str) -> list[dict[str, Any]]:
    if not token_spans:
        return []
    merged: list[dict[str, Any]] = []
    for start, end, probability in token_spans:
        start = max(0, int(start))
        end = min(len(output), int(end))
        if not merged or start > int(merged[-1]["end"]) + 1:
            merged.append({
                "start": start,
                "end": end,
                "text": output[start:end],
                "confidence": float(probability),
                "label_type": "modernbert_token_classifier",
            })
        else:
            merged[-1]["end"] = max(int(merged[-1]["end"]), end)
            merged[-1]["text"] = output[int(merged[-1]["start"]) : int(merged[-1]["end"])]
            merged[-1]["confidence"] = max(float(merged[-1]["confidence"]), float(probability))
    return merged


Overwriting tool_hallucination_detection/modeling.py


**Explanation:** writes the experiment orchestration layer. It combines dataset preparation, baselines, weighted ModernBERT training, separate validation thresholds for sentence and span decisions, per-type evaluation, and result export.


In [13]:
%%writefile tool_hallucination_detection/facade.py
"""Notebook-facing facade API."""
from __future__ import annotations

from pathlib import Path
import warnings
from typing import Any, Mapping, Sequence

from .baselines import (
    ImprovedEnsembleDetector,
    AttentionLookBackLensBaseline,
    LookBackLensStyleBaseline,
    RandomPriorBaseline,
    always_clean_predict,
    always_hallucinated_predict,
    random_balanced_predict,
    keyword_tool_action_predict,
    lettucedetect_predict,
    nli_sentence_verifier_predict,
)
from .corruption import build_corrupted_dataset
from .data import (
    flatten_splits,
    load_toolace_rows,
    normalize_toolace_rows,
    read_jsonl,
    split_by_source_id,
    synthetic_toolace_records,
    write_jsonl,
)
from .metrics import choose_best_threshold, evaluate_predictions, per_type_metrics, sentence_metrics


def prepare_dataset(
    quick: bool = False,
    seed: int = 42,
    cache_dir: str | Path | None = "data/processed",
    max_base_records: int | None = None,
    allow_synthetic_fallback: bool | None = None,
) -> dict[str, list[dict[str, Any]]]:
    """Prepare clean/corrupted RAGTruth-style records."""
    cache_path = Path(cache_dir) if cache_dir else None
    if cache_path and not quick:
        cached = _try_read_cached_splits(cache_path)
        if cached is not None and not _is_synthetic_dataset(cached):
            return cached

    if allow_synthetic_fallback is None:
        allow_synthetic_fallback = quick

    if quick:
        base_records = synthetic_toolace_records()
    else:
        try:
            rows = load_toolace_rows(max_records=max_base_records)
            base_records = normalize_toolace_rows(rows, max_records=max_base_records)
        except Exception:
            if not allow_synthetic_fallback:
                raise
            base_records = synthetic_toolace_records()

    if max_base_records is not None:
        base_records = base_records[:max_base_records]

    corrupted = build_corrupted_dataset(base_records, include_clean=True)
    splits = split_by_source_id(corrupted, seed=seed)
    if cache_path and not quick:
        for split, records in splits.items():
            write_jsonl(records, cache_path / f"{split}.jsonl")
    return splits


def run_baselines(
    quick: bool = False,
    dataset: dict[str, list[dict[str, Any]]] | None = None,
    include_lettucedetect_fallback: bool = False,
    require_real_lettucedetect: bool = False,
    lettuce_model_path: str | None = None,
    include_attention_lookback: bool = True,
    require_attention_lookback: bool = False,
    attention_lookback_model_name: str = "distilgpt2",
    attention_lookback_train_limit: int = 240,
    attention_lookback_max_length: int = 384,
    attention_lookback_max_answer_tokens: int = 96,
) -> dict[str, Any]:
    dataset = dataset or prepare_dataset(quick=quick)
    train_records = _non_empty_split(dataset, "train")
    validation_records = _non_empty_split(dataset, "validation")
    test_records = _non_empty_split(dataset, "test")

    rows = []
    predictions_by_method = {}
    thresholds_by_method = {}
    availability_rows = []

    methods: dict[str, Any] = {
        "always_clean": always_clean_predict,
        "always_hallucinated": always_hallucinated_predict,
        "random_balanced": random_balanced_predict,
        "random_prior": RandomPriorBaseline().fit(validation_records).predict,
        "keyword_tool_action": keyword_tool_action_predict,
    }

    # Lexical proxy is kept as a sanity baseline, not as the main LookBackLens result.
    lookback_proxy = LookBackLensStyleBaseline().fit(validation_records)
    methods["lookback_lens_style_proxy"] = lookback_proxy.predict

    if include_attention_lookback and not quick:
        try:
            attention_lookback = AttentionLookBackLensBaseline(
                model_name=attention_lookback_model_name,
                max_train_records=attention_lookback_train_limit,
                max_length=attention_lookback_max_length,
                max_answer_tokens=attention_lookback_max_answer_tokens,
            ).fit(train_records)
            methods["attention_lookback_lens_adapted"] = attention_lookback.predict
        except Exception as exc:
            availability_rows.append({"method": "attention_lookback_lens_adapted", "status": "unavailable", "details": repr(exc)})
            if require_attention_lookback:
                raise

    if not quick:
        methods["nli_sentence_verifier"] = nli_sentence_verifier_predict
        methods["lettucedetect_real"] = lambda records: lettucedetect_predict(
            records,
            model_path=lettuce_model_path,
            fallback_to_keyword=include_lettucedetect_fallback,
        )

    for method_name, predict_fn in methods.items():
        try:
            validation_predictions = predict_fn(validation_records)
            predictions = predict_fn(test_records)
        except Exception as exc:
            availability_rows.append({"method": method_name, "status": "unavailable", "details": repr(exc)})
            if method_name == "lettucedetect_real" and require_real_lettucedetect:
                raise
            if method_name == "attention_lookback_lens_adapted" and require_attention_lookback:
                raise
            continue

        validation_scores = [float(prediction.get("score", 0.0)) for prediction in validation_predictions]
        validation_gold = [int(len(record.get("labels", [])) > 0) for record in validation_records]
        threshold = 0.5 if method_name == "always_clean" else choose_best_threshold(validation_gold, validation_scores, metric="macro_f1")

        actual_method = str(predictions[0].get("method", method_name)) if predictions else method_name
        predictions_by_method[actual_method] = predictions
        evaluated = evaluate_predictions(test_records, predictions, threshold=threshold)
        thresholds_by_method[actual_method] = threshold
        row = {"method": actual_method, **{f"sentence_{k}": v for k, v in evaluated["sentence"].items()}}
        row.update({f"span_{k}": v for k, v in evaluated["span"].items()})
        rows.append(row)
        if "fallback" in actual_method:
            status = "fallback"
        elif actual_method == "lettucedetect_real":
            status = "real"
        elif actual_method == "attention_lookback_lens_adapted":
            status = "attention_adapted"
        else:
            status = "ok"
        availability_rows.append({"method": actual_method, "status": status, "details": ""})

    return {
        "records": test_records,
        "metrics": _to_table(rows),
        "predictions": predictions_by_method,
        "thresholds": thresholds_by_method,
        "availability": _to_table(availability_rows),
    }


def train_best_model(
    quick: bool = False,
    dataset: dict[str, list[dict[str, Any]]] | None = None,
    output_dir: str | Path = "artifacts/modernbert-token-classifier",
    strict: bool = False,
    model_name: str = "answerdotai/ModernBERT-base",
    epochs: float = 1.0,
    batch_size: int = 2,
    max_length: int = 2048,
    use_class_weights: bool = True,
) -> ImprovedEnsembleDetector | Path:
    """Train the best available detector.

    Quick mode returns a fitted lightweight ensemble. Full mode attempts to
    fine-tune the weighted transformer token classifier and falls back to the
    ensemble when strict=False.
    """
    dataset = dataset or prepare_dataset(quick=quick)
    validation_records = _non_empty_split(dataset, "validation")

    if quick:
        return ImprovedEnsembleDetector().fit(validation_records)

    try:
        from .modeling import train_token_classifier
        return train_token_classifier(
            train_records=_non_empty_split(dataset, "train"),
            validation_records=validation_records,
            output_dir=output_dir,
            model_name=model_name,
            epochs=epochs,
            batch_size=batch_size,
            max_length=max_length,
            use_class_weights=use_class_weights,
        )
    except Exception as exc:
        if strict:
            raise
        detector = ImprovedEnsembleDetector().fit(validation_records)
        setattr(detector, "training_error", repr(exc))
        warnings.warn(
            "Token-classifier training failed; falling back to the lightweight ensemble. "
            f"Original error: {exc!r}",
            RuntimeWarning,
            stacklevel=2,
        )
        return detector


def predict_spans(
    records: Sequence[Mapping[str, Any]],
    model_path: str | Path | ImprovedEnsembleDetector | None = None,
    span_threshold: float = 0.5,
    max_length: int = 2048,
) -> list[dict[str, Any]]:
    if isinstance(model_path, ImprovedEnsembleDetector):
        return model_path.predict(records)
    if model_path is not None:
        from .modeling import predict_with_token_classifier
        return predict_with_token_classifier(records, model_path, threshold=span_threshold, max_length=max_length)
    return ImprovedEnsembleDetector().fit(records).predict(records)


def evaluate_experiment(
    quick: bool = False,
    model_path: str | Path | None = None,
    seed: int = 42,
    max_base_records: int | None = None,
    strict_training: bool = False,
    model_name: str = "answerdotai/ModernBERT-base",
    epochs: float = 1.0,
    batch_size: int = 2,
    max_length: int = 2048,
    cache_dir: str | Path | None = "data/processed",
    use_class_weights: bool = True,
    include_lettucedetect_fallback: bool = False,
    require_real_lettucedetect: bool = False,
    lettuce_model_path: str | None = None,
    include_attention_lookback: bool = True,
    require_attention_lookback: bool = False,
    attention_lookback_model_name: str = "distilgpt2",
    attention_lookback_train_limit: int = 240,
    attention_lookback_max_length: int = 384,
    attention_lookback_max_answer_tokens: int = 96,
) -> dict[str, Any]:
    dataset = prepare_dataset(quick=quick, seed=seed, max_base_records=max_base_records, cache_dir=cache_dir)
    test_records = _non_empty_split(dataset, "test")
    validation_records = _non_empty_split(dataset, "validation")

    baseline_result = run_baselines(
        quick=quick,
        dataset=dataset,
        include_lettucedetect_fallback=include_lettucedetect_fallback,
        require_real_lettucedetect=require_real_lettucedetect,
        lettuce_model_path=lettuce_model_path,
        include_attention_lookback=include_attention_lookback,
        require_attention_lookback=require_attention_lookback,
        attention_lookback_model_name=attention_lookback_model_name,
        attention_lookback_train_limit=attention_lookback_train_limit,
        attention_lookback_max_length=attention_lookback_max_length,
        attention_lookback_max_answer_tokens=attention_lookback_max_answer_tokens,
    )

    if model_path is not None:
        detector_or_path: ImprovedEnsembleDetector | Path = Path(model_path)
    else:
        detector_or_path = train_best_model(
            quick=quick,
            dataset=dataset,
            strict=strict_training,
            model_name=model_name,
            epochs=epochs,
            batch_size=batch_size,
            max_length=max_length,
            use_class_weights=use_class_weights,
        )

    threshold_info: dict[str, float]
    if isinstance(detector_or_path, ImprovedEnsembleDetector):
        improved_sentence_threshold = detector_or_path.threshold
        improved_span_threshold = detector_or_path.threshold
        threshold_info = {
            "sentence_threshold": float(improved_sentence_threshold),
            "span_threshold": float(improved_span_threshold),
        }
    else:
        from .modeling import tune_token_classifier_thresholds
        threshold_info = tune_token_classifier_thresholds(
            validation_records=validation_records,
            model_path=detector_or_path,
            max_length=max_length,
            sentence_metric="macro_f1",
            span_metric="char_f1",
        )
        improved_sentence_threshold = float(threshold_info["sentence_threshold"])
        improved_span_threshold = float(threshold_info["span_threshold"])

    improved_predictions = predict_spans(
        test_records,
        detector_or_path,
        span_threshold=improved_span_threshold,
        max_length=max_length,
    )
    improved_eval = evaluate_predictions(test_records, improved_predictions, threshold=improved_sentence_threshold)

    rows = _table_to_rows(baseline_result["metrics"])
    improved_method = "improved_ensemble" if isinstance(detector_or_path, ImprovedEnsembleDetector) else "modernbert_token_classifier"
    improved_row = {
        "method": improved_method,
        **{f"sentence_{k}": v for k, v in improved_eval["sentence"].items()},
        **{f"span_{k}": v for k, v in improved_eval["span"].items()},
    }
    rows.append(improved_row)

    validation_scores = [prediction["score"] for prediction in ImprovedEnsembleDetector().fit(validation_records).predict(validation_records)]
    validation_gold = [int(len(record.get("labels", [])) > 0) for record in validation_records]
    validation_summary = sentence_metrics(validation_gold, validation_scores, choose_best_threshold(validation_gold, validation_scores, metric="macro_f1"))

    thresholds = {**baseline_result.get("thresholds", {}), improved_method: improved_sentence_threshold}
    thresholds[f"{improved_method}__sentence"] = improved_sentence_threshold
    thresholds[f"{improved_method}__span"] = improved_span_threshold

    return {
        "dataset": dataset,
        "test_records": test_records,
        "sentence_metrics": _to_table([{key: row[key] for key in row if key.startswith("sentence_") or key == "method"} for row in rows]),
        "span_metrics": _to_table([{key: row[key] for key in row if key.startswith("span_") or key == "method"} for row in rows]),
        "per_type_metrics": _to_table(per_type_metrics(test_records, improved_predictions, threshold=improved_sentence_threshold)),
        "all_metrics": _to_table(rows),
        "predictions": {"improved": improved_predictions, **baseline_result["predictions"]},
        "validation_summary": validation_summary,
        "thresholds": thresholds,
        "threshold_info": threshold_info,
        "baseline_availability": baseline_result.get("availability"),
        "training_error": getattr(detector_or_path, "training_error", None),
        "model_path": str(detector_or_path) if isinstance(detector_or_path, Path) else None,
    }


def export_splits(dataset: Mapping[str, list[dict[str, Any]]], output_dir: str | Path = "artifacts/ragtruth_toolace") -> dict[str, str]:
    output_dir = Path(output_dir)
    paths = {}
    for split, records in dataset.items():
        path = write_jsonl(records, output_dir / f"{split}.jsonl")
        paths[split] = str(path)
    return paths


def _try_read_cached_splits(cache_path: Path) -> dict[str, list[dict[str, Any]]] | None:
    paths = {split: cache_path / f"{split}.jsonl" for split in ("train", "validation", "test")}
    if not all(path.exists() for path in paths.values()):
        return None
    return {split: read_jsonl(path) for split, path in paths.items()}


def _is_synthetic_dataset(dataset: Mapping[str, list[dict[str, Any]]]) -> bool:
    records = flatten_splits(dataset)
    if not records:
        return False
    return all(str(record.get("source_id", "")).startswith("synthetic-") for record in records)


def _non_empty_split(dataset: Mapping[str, list[dict[str, Any]]], split: str) -> list[dict[str, Any]]:
    records = list(dataset.get(split, []))
    if records:
        return records
    return flatten_splits(dataset)


def _to_table(rows: list[dict[str, Any]]) -> Any:
    try:
        import pandas as pd
        return pd.DataFrame(rows)
    except Exception:
        return rows


def _table_to_rows(table: Any) -> list[dict[str, Any]]:
    if hasattr(table, "to_dict"):
        return list(table.to_dict(orient="records"))
    return list(table)


Overwriting tool_hallucination_detection/facade.py


**Explanation:** exposes a small public API (`prepare_dataset`, `evaluate_experiment`, etc.) so the notebook can be used similarly to the original GitHub package.


In [14]:
%%writefile tool_hallucination_detection/__init__.py
from .facade import (
    prepare_dataset,
    run_baselines,
    train_best_model,
    predict_spans,
    evaluate_experiment,
    export_splits,
)

__all__ = [
    "prepare_dataset",
    "run_baselines",
    "train_best_model",
    "predict_spans",
    "evaluate_experiment",
    "export_splits",
]


Overwriting tool_hallucination_detection/__init__.py


## 3.3 Smoke test on a tiny synthetic dataset

This cell checks that the local package imports and all lightweight paths work before downloading ToolACE/models.


**Explanation:** runs a tiny synthetic smoke test before downloading large datasets/models. If this fails, the later full run should not be started.


In [15]:
from tool_hallucination_detection import evaluate_experiment, prepare_dataset

quick_result = evaluate_experiment(quick=True, cache_dir=None)
print({split: len(records) for split, records in quick_result["dataset"].items()})
quick_result["all_metrics"]


{'train': 4, 'validation': 4, 'test': 4}


,method,sentence_n,sentence_positive_rate,sentence_predicted_positive_rate,sentence_accuracy,sentence_balanced_accuracy,sentence_precision,sentence_recall,sentence_f1,sentence_negative_precision,...,sentence_tn,span_char_precision,span_char_recall,span_char_f1,span_exact_span_precision,span_exact_span_recall,span_exact_span_f1,span_relaxed_span_precision,span_relaxed_span_recall,span_relaxed_span_f1
0,always_clean,4.0,0.75,0.00,0.25,0.500000,0.000000,0.000000,0.000000,0.250000,...,1.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
1,always_hallucinated,4.0,0.75,1.00,0.75,0.500000,0.750000,1.000000,0.857143,0.000000,...,0.0,0.369272,1.000000,0.539370,0.000000,0.000000,0.000000,0.250000,0.333333,0.285714
2,random_balanced,4.0,0.75,0.75,0.50,0.333333,0.666667,0.666667,0.666667,0.000000,...,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
3,random_prior,4.0,0.75,0.50,0.25,0.166667,0.500000,0.333333,0.400000,0.000000,...,0.0,0.016949,0.014599,0.015686,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
4,keyword_tool_action,4.0,0.75,0.25,0.50,0.666667,1.000000,0.333333,0.500000,0.333333,...,1.0,1.000000,0.627737,0.771300,1.000000,0.333333,0.500000,1.000000,0.333333,0.500000
5,lookback_lens_style_proxy,4.0,0.75,1.00,0.75,0.500000,0.750000,1.000000,0.857143,0.000000,...,0.0,0.369272,1.000000,0.539370,0.333333,0.666667,0.444444,0.333333,0.666667,0.444444
6,improved_ensemble,4.0,0.75,1.00,0.75,0.500000,0.750000,1.000000,0.857143,0.000000,...,0.0,0.439103,1.000000,0.610245,0.250000,0.333333,0.285714,0.250000,0.333333,0.285714


## 3.4 Download and prepare ToolACE-based RAGTruth-style data

Set `MAX_BASE_RECORDS=None` for a larger/full ToolACE run. The value `200` reproduces the smaller Kaggle/Colab run you were trying.


**Explanation:** loads ToolACE, creates the clean and corrupted examples, and prints split/type counts. This is the first real dataset construction step.


In [16]:
from tool_hallucination_detection import prepare_dataset, export_splits

MAX_BASE_RECORDS = 200
CACHE_DIR = None  # None avoids accidentally reusing stale cached data

dataset = prepare_dataset(
    quick=False,
    max_base_records=MAX_BASE_RECORDS,
    cache_dir=CACHE_DIR,
)

split_sizes = {split: len(records) for split, records in dataset.items()}
print("Split sizes:", split_sizes)

all_records = [record for records in dataset.values() for record in records]
counts = pd.DataFrame(all_records).groupby(["split", "corruption_type"]).size().reset_index(name="n")
counts


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Split sizes: {'train': 556, 'validation': 120, 'test': 120}


,split,corruption_type,n
0,test,clean,30
1,test,missing_tool,30
2,test,overgeneration,30
3,test,tool_contradiction,30
4,train,clean,139
5,train,missing_tool,139
6,train,overgeneration,139
7,train,tool_contradiction,139
8,validation,clean,30
9,validation,missing_tool,30


**Explanation:** exports the generated dataset in RAGTruth-like JSONL format, including per-type subsets for contradiction, overgeneration, and missing-tool hallucinations.


In [17]:
# Save generated span-labeled dataset locally in RAGTruth-like JSONL format.
exported_paths = export_splits(dataset, "artifacts/ragtruth_toolace")
exported_paths


# Also export the three required per-type datasets with clean controls.
# This makes the "three datasets with different hallucination types" requirement explicit.
PER_TYPE_DATASETS = {}
for hallucination_type in ["tool_contradiction", "overgeneration", "missing_tool"]:
    filtered = {
        split: [
            record for record in records
            if record["corruption_type"] in {"clean", hallucination_type}
        ]
        for split, records in dataset.items()
    }
    PER_TYPE_DATASETS[hallucination_type] = export_splits(
        filtered,
        output_dir=f"artifacts/ragtruth_toolace/by_type/{hallucination_type}",
    )

print("Per-type dataset exports:")
PER_TYPE_DATASETS


Per-type dataset exports:


{'tool_contradiction': {'train': 'artifacts/ragtruth_toolace/by_type/tool_contradiction/train.jsonl',
  'validation': 'artifacts/ragtruth_toolace/by_type/tool_contradiction/validation.jsonl',
  'test': 'artifacts/ragtruth_toolace/by_type/tool_contradiction/test.jsonl'},
 'overgeneration': {'train': 'artifacts/ragtruth_toolace/by_type/overgeneration/train.jsonl',
  'validation': 'artifacts/ragtruth_toolace/by_type/overgeneration/validation.jsonl',
  'test': 'artifacts/ragtruth_toolace/by_type/overgeneration/test.jsonl'},
 'missing_tool': {'train': 'artifacts/ragtruth_toolace/by_type/missing_tool/train.jsonl',
  'validation': 'artifacts/ragtruth_toolace/by_type/missing_tool/validation.jsonl',
  'test': 'artifacts/ragtruth_toolace/by_type/missing_tool/test.jsonl'}}

## 3.5 Automated dataset and label checks

This section checks the parts that make the submission credible: no split leakage, valid span offsets, all required RAGTruth-style fields, correct binary labels, and non-trivial class balance. These checks are also useful for the report because they show that the numbers are computed on a clean evaluation setup rather than on leaked or malformed data.


**Explanation:** performs automatic dataset validation: required fields, valid offsets, split leakage, label consistency, and binary class balance.


In [18]:
from collections import Counter, defaultdict
from IPython.display import display
from tool_hallucination_detection.schema import sentence_label, validate_labels

REQUIRED_FIELDS = {"id", "source_id", "split", "corruption_type", "query", "context", "output", "labels", "hallucination_labels"}
HALLUCINATION_TYPES = {"tool_contradiction", "overgeneration", "missing_tool"}


def audit_dataset(dataset):
    rows = []
    hard_failures = []
    split_to_source_ids = {}

    for split, records in dataset.items():
        split_to_source_ids[split] = {str(record.get("source_id")) for record in records}
        counts = Counter(record.get("corruption_type", "unknown") for record in records)
        rows.extend({"split": split, "corruption_type": key, "n": value} for key, value in sorted(counts.items()))

        for record in records:
            missing = REQUIRED_FIELDS - set(record)
            if missing:
                hard_failures.append(f"{record.get('id')}: missing fields {sorted(missing)}")
            try:
                validate_labels(record)
            except Exception as exc:
                hard_failures.append(f"{record.get('id')}: invalid span labels: {exc}")
            if record.get("corruption_type") == "clean" and record.get("labels"):
                hard_failures.append(f"{record.get('id')}: clean example has hallucination labels")
            if record.get("corruption_type") in HALLUCINATION_TYPES and not record.get("labels"):
                hard_failures.append(f"{record.get('id')}: corrupted example has no hallucination labels")
            for label in record.get("labels", []):
                span_text = record["output"][int(label["start"]):int(label["end"])]
                if span_text != label.get("text"):
                    hard_failures.append(f"{record.get('id')}: label text does not match output slice")

    leakage_rows = []
    for left in split_to_source_ids:
        for right in split_to_source_ids:
            if left >= right:
                continue
            overlap = split_to_source_ids[left] & split_to_source_ids[right]
            leakage_rows.append({"split_a": left, "split_b": right, "source_id_overlap": len(overlap)})
            if overlap:
                hard_failures.append(f"source_id leakage between {left} and {right}: {sorted(list(overlap))[:5]}")

    all_records = [record for records in dataset.values() for record in records]
    binary = [sentence_label(record) for record in all_records]
    class_balance = pd.DataFrame([{
        "n_total": len(all_records),
        "n_clean": int(sum(1 for value in binary if value == 0)),
        "n_hallucinated": int(sum(1 for value in binary if value == 1)),
        "positive_rate": float(np.mean(binary)) if binary else np.nan,
    }])

    return {
        "split_type_counts": pd.DataFrame(rows),
        "leakage": pd.DataFrame(leakage_rows),
        "class_balance": class_balance,
        "hard_failures": hard_failures,
    }


audit = audit_dataset(dataset)
print("Split/type counts:")
display(audit["split_type_counts"])
print("Source leakage check:")
display(audit["leakage"])
print("Binary class balance:")
display(audit["class_balance"])

if audit["hard_failures"]:
    print("Hard failures:")
    for item in audit["hard_failures"][:20]:
        print("-", item)
    raise AssertionError(f"Dataset audit failed with {len(audit['hard_failures'])} issue(s).")
else:
    print("All dataset and label checks passed.")


Split/type counts:


,split,corruption_type,n
0,train,clean,139
1,train,missing_tool,139
2,train,overgeneration,139
3,train,tool_contradiction,139
4,validation,clean,30
5,validation,missing_tool,30
6,validation,overgeneration,30
7,validation,tool_contradiction,30
8,test,clean,30
9,test,missing_tool,30


Source leakage check:


,split_a,split_b,source_id_overlap
0,train,validation,0
1,test,train,0
2,test,validation,0


Binary class balance:


,n_total,n_clean,n_hallucinated,positive_rate
0,796,199,597,0.75


All dataset and label checks passed.


### Why trivial baselines are necessary

Because the generated data usually contains one clean variant and three hallucinated variants per source dialogue, the positive class can be around 75%. In such a setting, a classifier that always predicts “hallucinated” may get a high positive-class F1. This is why the final comparison focuses on `sentence_macro_f1`, `sentence_balanced_accuracy`, and the confusion matrix, not only on positive-class F1.


**Explanation:** demonstrates why trivial baselines are necessary. It shows that an `always_hallucinated` classifier can look deceptively strong under positive-class F1.


In [19]:
from tool_hallucination_detection.metrics import sentence_metrics
from tool_hallucination_detection.schema import sentence_label

test_like_records = dataset["test"]
gold_binary = [sentence_label(record) for record in test_like_records]

sanity_rows = []
for name, scores in {
    "always_clean": [0.0] * len(gold_binary),
    "always_hallucinated": [1.0] * len(gold_binary),
    "constant_prevalence_score": [float(np.mean(gold_binary))] * len(gold_binary),
}.items():
    m = sentence_metrics(gold_binary, scores, threshold=0.5)
    sanity_rows.append({"baseline": name, **m})

sanity_df = pd.DataFrame(sanity_rows)
display(sanity_df[[
    "baseline", "positive_rate", "accuracy", "balanced_accuracy", "f1", "negative_f1", "macro_f1", "tp", "fp", "fn", "tn"
]])


,baseline,positive_rate,accuracy,balanced_accuracy,f1,negative_f1,macro_f1,tp,fp,fn,tn
0,always_clean,0.75,0.25,0.5,0.000000,0.4,0.200000,0.0,0.0,90.0,30.0
1,always_hallucinated,0.75,0.75,0.5,0.857143,0.0,0.428571,90.0,30.0,0.0,0.0
2,constant_prevalence_score,0.75,0.75,0.5,0.857143,0.0,0.428571,90.0,30.0,0.0,0.0


## 3.6 Inspect span labels

**What happens here:** we print representative examples for each hallucination type and visually check that the gold character spans point to the intended hallucinated parts of the answer. This is important because span-level metrics are only meaningful if the offsets are correct.


**Explanation:** prints qualitative examples and gold spans. Use this cell to verify that the synthetic hallucinations are correctly placed and labeled.


In [20]:
# Show one example per corruption type with gold spans.
examples = []
seen_types = set()
for record in all_records:
    ctype = record["corruption_type"]
    if ctype not in seen_types:
        examples.append(record)
        seen_types.add(ctype)
    if len(seen_types) == 4:
        break

for record in examples:
    print("=" * 100)
    print("ID:", record["id"])
    print("TYPE:", record["corruption_type"])
    print("QUERY:", record["query"][:500])
    print("CONTEXT:", record["context"][:500])
    print("OUTPUT:", record["output"][:800])
    print("LABELS:", record["labels"])


ID: toolace-158::clean
TYPE: clean
QUERY: I want to see the videos of user with uid: 123456 on Likee. Can you fetch them for me?
CONTEXT: [{"name": "Get User Videos", "results": {"videos": [{"videoId": "vid123", "title": "Beach Day", "description": "Having fun at the beach with friends!", "thumbnail": "http://example.com/thumbnail1.jpg", "views": 150}, {"videoId": "vid124", "title": "Cooking Pasta", "description": "A quick tutorial on making delicious pasta", "thumbnail": "http://example.com/thumbnail2.jpg", "views": 200}, {"videoId": "vid125", "title": "Workout Routine", "description": "My daily fitness routine to stay healthy", 
OUTPUT: Here are the videos of the user with UID 123456 on Likee: 1. **Title:** Beach Day - **Description:** Having fun at the beach with friends! - **Thumbnail:** ![Thumbnail](http://example.com/thumbnail1.jpg) - **Views:** 150 2. **Title:** Cooking Pasta - **Description:** A quick tutorial on making delicious pasta - **Thumbnail:** ![Thumbnail](http://examp

### Dataset hardness audit

The next cell summarizes which corruption styles were actually generated. This is important because a benchmark can look good but still be too easy if every hallucination is appended with the same phrase. We therefore inspect whether contradictions include short substitutions and whether overgeneration/missing-tool examples use diverse templates.


In [21]:
# Summarize corruption styles from label metadata.
import re

style_rows = []
for split, records in dataset.items():
    for record in records:
        labels = record.get("labels", [])
        if not labels:
            style_rows.append({
                "split": split,
                "corruption_type": record.get("corruption_type"),
                "style": "clean",
                "span_length": 0,
                "inserted_at_end": False,
            })
        for label in labels:
            meta = str(label.get("meta", ""))
            match = re.search(r"style=([^;]+)", meta)
            style = match.group(1) if match else "unknown"
            start = int(label.get("start", 0))
            end = int(label.get("end", start))
            style_rows.append({
                "split": split,
                "corruption_type": record.get("corruption_type"),
                "style": style,
                "span_length": max(0, end - start),
                "inserted_at_end": end == len(str(record.get("output", ""))),
            })

hardness_df = pd.DataFrame(style_rows)
print("Corruption style counts:")
display(hardness_df.groupby(["corruption_type", "style"]).size().reset_index(name="n"))
print("Average hallucinated span length by type:")
display(hardness_df[hardness_df["corruption_type"] != "clean"].groupby("corruption_type")["span_length"].agg(["count", "mean", "median", "min", "max"]).reset_index())
print("Share of spans inserted at the end of the answer:")
display(hardness_df[hardness_df["corruption_type"] != "clean"].groupby("corruption_type")["inserted_at_end"].mean().reset_index(name="share_inserted_at_end"))


Corruption style counts:


,corruption_type,style,n
0,clean,clean,199
1,missing_tool,calendar,40
2,missing_tool,cancel,51
3,missing_tool,email,49
4,missing_tool,flight,38
5,missing_tool,hotel,16
6,missing_tool,payment,2
7,missing_tool,ride,3
8,overgeneration,unsupported_cause,20
9,overgeneration,unsupported_comparison,15


Average hallucinated span length by type:


,corruption_type,count,mean,median,min,max
0,missing_tool,199,47.035176,50.0,38,54
1,overgeneration,199,73.587940,72.0,61,86
2,tool_contradiction,199,19.110553,6.0,1,85


Share of spans inserted at the end of the answer:


,corruption_type,share_inserted_at_end
0,missing_tool,0.733668
1,overgeneration,0.743719
2,tool_contradiction,0.115578


**How to read this audit:** a stronger synthetic benchmark should not consist only of end-appended long spans. Some end-appended spans are expected for overgeneration and missing-tool actions, but `tool_contradiction` should ideally include many short substitutions such as number/date/status/entity changes. If all contradictions fall into `fallback_*`, the source answers did not copy enough tool values and the contradiction generation is weaker for that run.


## 3.7 Run baselines

This evaluates the required baseline slots. `lettucedetect` runs only if installed/available; otherwise the notebook uses a deterministic fallback so that `Run all` stays reproducible.


**Explanation:** evaluates baseline methods on the same test split that will be used for the improved model.


In [22]:
# Baselines are executed as part of evaluate_experiment below.
# This standalone cell is intentionally skipped to avoid downloading external models twice.
print("Baselines will be run in the final experiment cell.")


Baselines will be run in the final experiment cell.


## 3.8 Train improved span detector and run final experiment

Default settings are chosen to run on a Colab GPU. If you only want a fast check, set `STRICT_TRAINING=False` and/or `MODEL_NAME="distilbert-base-uncased"`.


**Explanation:** runs the full experiment: dataset preparation, baselines, ModernBERT training, validation-threshold selection, and test evaluation.


In [23]:
from tool_hallucination_detection import evaluate_experiment

STRICT_TRAINING = True
MODEL_NAME = "answerdotai/ModernBERT-base"       # repo/default model
# MODEL_NAME = "distilbert-base-uncased"          # faster fallback if ModernBERT is too slow
EPOCHS = 1.0
BATCH_SIZE = 2
MAX_LENGTH = 2048

# External assignment baselines.
# LettuceDetect is required by the assignment. We do not silently replace it with a keyword fallback.
REQUIRE_REAL_LETTUCEDETECT = True
LETTUCEDETECT_MODEL_PATH = "KRLabsOrg/lettucedect-base-modernbert-en-v1"

# Adapted LookBackLens baseline.
# distilgpt2 is small enough for Colab; increase train limit/model size only if you have spare GPU time.
INCLUDE_ATTENTION_LOOKBACK = True
REQUIRE_ATTENTION_LOOKBACK = True
ATTENTION_LOOKBACK_MODEL = "distilgpt2"
ATTENTION_LOOKBACK_TRAIN_LIMIT = 240
ATTENTION_LOOKBACK_MAX_LENGTH = 384
ATTENTION_LOOKBACK_MAX_ANSWER_TOKENS = 96

full_result = evaluate_experiment(
    quick=False,
    max_base_records=MAX_BASE_RECORDS,
    strict_training=STRICT_TRAINING,
    model_name=MODEL_NAME,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    max_length=MAX_LENGTH,
    cache_dir=CACHE_DIR,
    use_class_weights=True,
    include_lettucedetect_fallback=False,
    require_real_lettucedetect=REQUIRE_REAL_LETTUCEDETECT,
    lettuce_model_path=LETTUCEDETECT_MODEL_PATH,
    include_attention_lookback=INCLUDE_ATTENTION_LOOKBACK,
    require_attention_lookback=REQUIRE_ATTENTION_LOOKBACK,
    attention_lookback_model_name=ATTENTION_LOOKBACK_MODEL,
    attention_lookback_train_limit=ATTENTION_LOOKBACK_TRAIN_LIMIT,
    attention_lookback_max_length=ATTENTION_LOOKBACK_MAX_LENGTH,
    attention_lookback_max_answer_tokens=ATTENTION_LOOKBACK_MAX_ANSWER_TOKENS,
)

print("Model path:", full_result.get("model_path"))
print("Training error:", full_result.get("training_error"))
full_result["all_metrics"]


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/599M [00:00<?, ?B/s]

Some weights of ModernBertForTokenClassification were not initialized from the model checkpoint at answerdotai/ModernBERT-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': None, 'bos_token_id': None}.


Epoch,Training Loss,Validation Loss
1,0.212300,0.115046


Model path: artifacts/modernbert-token-classifier
Training error: None


,method,sentence_n,sentence_positive_rate,sentence_predicted_positive_rate,sentence_accuracy,sentence_balanced_accuracy,sentence_precision,sentence_recall,sentence_f1,sentence_negative_precision,...,sentence_tn,span_char_precision,span_char_recall,span_char_f1,span_exact_span_precision,span_exact_span_recall,span_exact_span_f1,span_relaxed_span_precision,span_relaxed_span_recall,span_relaxed_span_f1
0,always_clean,120.0,0.75,0.000000,0.250000,0.500000,0.000000,0.000000,0.000000,0.250000,...,30.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
1,always_hallucinated,120.0,0.75,1.000000,0.750000,0.500000,0.750000,1.000000,0.857143,0.000000,...,0.0,0.073248,1.000000,0.136498,0.000000,0.000000,0.000000,0.008333,0.011111,0.009524
2,random_balanced,120.0,0.75,0.891667,0.658333,0.450000,0.728972,0.866667,0.791878,0.076923,...,1.0,0.066451,0.568187,0.118986,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
3,random_prior,120.0,0.75,0.583333,0.483333,0.422222,0.700000,0.544444,0.612500,0.180000,...,9.0,0.070427,0.542393,0.124667,0.000000,0.000000,0.000000,0.014286,0.011111,0.012500
4,keyword_tool_action,120.0,0.75,0.658333,0.641667,0.583333,0.797468,0.700000,0.745562,0.341463,...,14.0,0.134319,0.672558,0.223919,0.104478,0.155556,0.125000,0.201493,0.300000,0.241071
5,lookback_lens_style_proxy,120.0,0.75,0.658333,0.625000,0.561111,0.784810,0.688889,0.733728,0.317073,...,13.0,0.883602,0.248388,0.387770,0.523810,0.122222,0.198198,0.952381,0.222222,0.360360
6,attention_lookback_lens_adapted,120.0,0.75,0.583333,0.566667,0.533333,0.771429,0.600000,0.675000,0.280000,...,14.0,0.058259,0.588966,0.106030,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
7,nli_sentence_verifier,120.0,0.75,0.833333,0.766667,0.633333,0.810000,0.900000,0.852632,0.550000,...,11.0,0.119055,0.929066,0.211063,0.073770,0.400000,0.124567,0.116803,0.633333,0.197232
8,lettucedetect_real,120.0,0.75,0.608333,0.591667,0.550000,0.780822,0.633333,0.699387,0.297872,...,14.0,0.091139,0.576546,0.157397,0.034146,0.077778,0.047458,0.073171,0.166667,0.101695
9,modernbert_token_classifier,120.0,0.75,0.575000,0.808333,0.861111,0.985507,0.755556,0.855346,0.568627,...,29.0,0.989039,0.926678,0.956843,0.625000,0.500000,0.555556,0.875000,0.700000,0.777778


### Baseline availability and threshold sanity check

**Explanation:** reports whether external baselines were actually available. In particular, a LettuceDetect fallback is explicitly marked as fallback and should not be described as a real LettuceDetect run in the final report. The cell also prints the separately tuned sentence/span thresholds for the improved model.

In [24]:
baseline_availability = full_result.get("baseline_availability")
if baseline_availability is not None:
    display(baseline_availability)
else:
    print("No baseline availability table was returned.")

print("Thresholds:")
for key, value in full_result.get("thresholds", {}).items():
    print(f"  {key}: {value:.4f}" if isinstance(value, float) else f"  {key}: {value}")

print("Detailed improved threshold info:", full_result.get("threshold_info", {}))


,method,status,details
0,always_clean,ok,
1,always_hallucinated,ok,
2,random_balanced,ok,
3,random_prior,ok,
4,keyword_tool_action,ok,
5,lookback_lens_style_proxy,ok,
6,attention_lookback_lens_adapted,attention_adapted,
7,nli_sentence_verifier,ok,
8,lettucedetect_real,real,


Thresholds:
  always_clean: 0.5000
  always_hallucinated: 0.5000
  random_balanced: 0.1072
  random_prior: 0.4666
  keyword_tool_action: 0.0000
  lookback_lens_style_proxy: 0.4681
  attention_lookback_lens_adapted: 0.4790
  nli_sentence_verifier: 0.6414
  lettucedetect_real: 0.9903
  modernbert_token_classifier: 0.8358
  modernbert_token_classifier__sentence: 0.8358
  modernbert_token_classifier__span: 0.9000
Detailed improved threshold info: {'sentence_threshold': 0.835761548088623, 'span_threshold': 0.9, 'validation_char_f1': 0.9560794044665012}


## 3.9 Main metric table and numerical checks

The following cells answer the key question: are the obtained metrics evidence of a good solution, or just numbers? We compare the improved method against trivial baselines and stronger baselines using macro-F1 and balanced accuracy. This is more reliable than looking only at positive-class F1.


**Explanation:** displays the main metric table. The best evidence of quality is macro-F1/balanced accuracy improvement over trivial and non-trivial baselines.


In [25]:
metrics_df = full_result["all_metrics"].copy()

preferred_cols = [
    "method",
    "sentence_accuracy",
    "sentence_balanced_accuracy",
    "sentence_precision",
    "sentence_recall",
    "sentence_f1",
    "sentence_negative_f1",
    "sentence_macro_f1",
    "sentence_roc_auc",
    "sentence_pr_auc",
    "sentence_brier",
    "sentence_ece_10",
    "span_char_f1",
    "span_exact_span_f1",
    "span_relaxed_span_f1",
]
existing_cols = [col for col in preferred_cols if col in metrics_df.columns]

sort_col = "sentence_macro_f1" if "sentence_macro_f1" in metrics_df.columns else "sentence_f1"
display(metrics_df[existing_cols].sort_values(sort_col, ascending=False).reset_index(drop=True))


,method,sentence_accuracy,sentence_balanced_accuracy,sentence_precision,sentence_recall,sentence_f1,sentence_negative_f1,sentence_macro_f1,sentence_roc_auc,sentence_pr_auc,sentence_brier,sentence_ece_10,span_char_f1,span_exact_span_f1,span_relaxed_span_f1
0,modernbert_token_classifier,0.808333,0.861111,0.985507,0.755556,0.855346,0.716049,0.785698,0.894630,0.968790,0.120988,0.082620,0.956843,0.555556,0.777778
1,nli_sentence_verifier,0.766667,0.633333,0.810000,0.900000,0.852632,0.440000,0.646316,0.654074,0.809387,0.194914,0.156217,0.211063,0.124567,0.197232
2,keyword_tool_action,0.641667,0.583333,0.797468,0.700000,0.745562,0.394366,0.569964,0.666296,0.846238,0.327292,0.305833,0.223919,0.125000,0.241071
3,lookback_lens_style_proxy,0.625000,0.561111,0.784810,0.688889,0.733728,0.366197,0.549962,0.642593,0.839647,0.216372,0.195039,0.387770,0.198198,0.360360
4,lettucedetect_real,0.591667,0.550000,0.780822,0.633333,0.699387,0.363636,0.531511,0.533704,0.774622,0.334531,0.332768,0.157397,0.047458,0.101695
5,attention_lookback_lens_adapted,0.566667,0.533333,0.771429,0.600000,0.675000,0.350000,0.512500,0.554630,0.779229,0.248545,0.248073,0.106030,0.000000,0.000000
6,always_hallucinated,0.750000,0.500000,0.750000,1.000000,0.857143,0.000000,0.428571,0.500000,0.750000,0.250000,0.250000,0.136498,0.000000,0.009524
7,random_balanced,0.658333,0.450000,0.728972,0.866667,0.791878,0.046512,0.419195,0.465185,0.750402,0.342379,0.302537,0.118986,0.000000,0.000000
8,random_prior,0.483333,0.422222,0.700000,0.544444,0.612500,0.225000,0.418750,0.371852,0.702970,0.364702,0.372072,0.124667,0.000000,0.012500
9,always_clean,0.250000,0.500000,0.000000,0.000000,0.000000,0.400000,0.200000,0.500000,0.750000,0.750000,0.750000,0.000000,0.000000,0.000000


### Confusion matrix by method

This table is often more informative than F1 alone. A method that predicts all examples as hallucinated will have high recall but many false positives on clean examples. A method that predicts all examples as clean will have high specificity but zero recall for hallucinations.


**Explanation:** shows confusion-matrix statistics by method. This catches degenerate detectors that predict only one class.


In [26]:
cm_cols = [
    "method",
    "sentence_positive_rate",
    "sentence_predicted_positive_rate",
    "sentence_tp",
    "sentence_fp",
    "sentence_fn",
    "sentence_tn",
    "sentence_recall",
    "sentence_specificity",
    "sentence_balanced_accuracy",
    "sentence_macro_f1",
]
cm_cols = [col for col in cm_cols if col in metrics_df.columns]
display(metrics_df[cm_cols])


,method,sentence_positive_rate,sentence_predicted_positive_rate,sentence_tp,sentence_fp,sentence_fn,sentence_tn,sentence_recall,sentence_specificity,sentence_balanced_accuracy,sentence_macro_f1
0,always_clean,0.75,0.000000,0.0,0.0,90.0,30.0,0.000000,1.000000,0.500000,0.200000
1,always_hallucinated,0.75,1.000000,90.0,30.0,0.0,0.0,1.000000,0.000000,0.500000,0.428571
2,random_balanced,0.75,0.891667,78.0,29.0,12.0,1.0,0.866667,0.033333,0.450000,0.419195
3,random_prior,0.75,0.583333,49.0,21.0,41.0,9.0,0.544444,0.300000,0.422222,0.418750
4,keyword_tool_action,0.75,0.658333,63.0,16.0,27.0,14.0,0.700000,0.466667,0.583333,0.569964
5,lookback_lens_style_proxy,0.75,0.658333,62.0,17.0,28.0,13.0,0.688889,0.433333,0.561111,0.549962
6,attention_lookback_lens_adapted,0.75,0.583333,54.0,16.0,36.0,14.0,0.600000,0.466667,0.533333,0.512500
7,nli_sentence_verifier,0.75,0.833333,81.0,19.0,9.0,11.0,0.900000,0.366667,0.633333,0.646316
8,lettucedetect_real,0.75,0.608333,57.0,16.0,33.0,14.0,0.633333,0.466667,0.550000,0.531511
9,modernbert_token_classifier,0.75,0.575000,68.0,1.0,22.0,29.0,0.755556,0.966667,0.861111,0.785698


### Automated verdict: does the improved method beat baselines?

This cell creates a conservative numerical interpretation. It does not claim that the model is universally good; it checks whether the improved method is better than trivial baselines and better than the strongest available non-improved baseline on this test split.


**Explanation:** automatically checks whether the improved method beats the best trivial baseline, the best non-improved baseline, and the assignment-style baselines.


In [50]:
def _pick_score_col(df):
    for col in ["sentence_macro_f1", "sentence_balanced_accuracy", "sentence_f1"]:
        if col in df.columns:
            return col
    raise ValueError("No suitable score column found")


def _as_float(value):
    try:
        return float(value)
    except Exception:
        return float("nan")


def build_verdict_table(metrics_df):
    df = metrics_df.copy()
    score_col = "sentence_macro_f1"
    df[score_col] = df[score_col].astype(float)

    improved_method = "modernbert_token_classifier"
    improved_score = float(df.loc[df["method"] == improved_method, score_col].iloc[0])

    trivial_methods = ["always_clean", "always_hallucinated", "random_balanced", "random_prior"]
    required_methods = ["lettucedetect_real", "attention_lookback_lens_adapted"]

    non_improved_df = df[df["method"] != improved_method]
    trivial_df = df[df["method"].isin(trivial_methods)]
    required_df = df[df["method"].isin(required_methods)]

    rows = []

    if len(trivial_df):
        best = trivial_df.sort_values(score_col, ascending=False).iloc[0]
        rows.append({
            "comparison": "improved vs best_trivial",
            "metric": score_col,
            "improved_method": improved_method,
            "improved_score": improved_score,
            "baseline_method": best["method"],
            "baseline_score": float(best[score_col]),
            "delta": improved_score - float(best[score_col]),
            "beats_baseline": improved_score > float(best[score_col]),
        })

    if len(non_improved_df):
        best = non_improved_df.sort_values(score_col, ascending=False).iloc[0]
        rows.append({
            "comparison": "improved vs best_non_improved",
            "metric": score_col,
            "improved_method": improved_method,
            "improved_score": improved_score,
            "baseline_method": best["method"],
            "baseline_score": float(best[score_col]),
            "delta": improved_score - float(best[score_col]),
            "beats_baseline": improved_score > float(best[score_col]),
        })

    if len(required_df):
        best = required_df.sort_values(score_col, ascending=False).iloc[0]
        rows.append({
            "comparison": "improved vs best_required_assignment_baseline",
            "metric": score_col,
            "improved_method": improved_method,
            "improved_score": improved_score,
            "baseline_method": best["method"],
            "baseline_score": float(best[score_col]),
            "delta": improved_score - float(best[score_col]),
            "beats_baseline": improved_score > float(best[score_col]),
        })

    return pd.DataFrame(rows)

verdict_df = build_verdict_table(metrics_df)
display(verdict_df)
verdict_df.to_csv("artifacts/verdict.csv", index=False)

# A compact human-readable summary.
for _, row in verdict_df.iterrows():
    sign = "beats" if row["beats_baseline"] else "does NOT beat"
    print(
        f"{row['improved_method']} {sign} {row['baseline_method']} by "
        f"{row['delta']:+.4f} on {row['metric']}."
    )


,comparison,metric,improved_method,improved_score,baseline_method,baseline_score,delta,beats_baseline
0,improved vs best_trivial,sentence_macro_f1,modernbert_token_classifier,0.785698,always_hallucinated,0.428571,0.357126,True
1,improved vs best_non_improved,sentence_macro_f1,modernbert_token_classifier,0.785698,nli_sentence_verifier,0.646316,0.139382,True
2,improved vs best_required_assignment_baseline,sentence_macro_f1,modernbert_token_classifier,0.785698,lettucedetect_real,0.531511,0.254186,True


modernbert_token_classifier beats always_hallucinated by +0.3571 on sentence_macro_f1.
modernbert_token_classifier beats nli_sentence_verifier by +0.1394 on sentence_macro_f1.
modernbert_token_classifier beats lettucedetect_real by +0.2542 on sentence_macro_f1.


### Course-inspired diagnostic table: quality vs. practicality

**What happens here:** this table does not change the score, but it strengthens the methodological discussion. In tool/RAG-style systems, a detector should be evaluated not only by accuracy but also by practical constraints: whether it requires training, whether it needs generation-time attention traces, and how expensive it is at inference time.


In [51]:
practicality_rows = [
    {
        "method_family": "trivial baseline",
        "example_methods": "always_clean, always_hallucinated, random",
        "needs_training": "no",
        "needs_generation_trace": "no",
        "inference_cost": "very low",
        "role_in_evaluation": "checks whether metrics are misleading under class imbalance",
    },
    {
        "method_family": "rule baseline",
        "example_methods": "keyword_tool_action",
        "needs_training": "no",
        "needs_generation_trace": "no",
        "inference_cost": "very low",
        "role_in_evaluation": "simple sanity check for missing-tool/action phrases",
    },
    {
        "method_family": "semantic verifier",
        "example_methods": "nli_sentence_verifier",
        "needs_training": "no",
        "needs_generation_trace": "no",
        "inference_cost": "medium",
        "role_in_evaluation": "checks whether answer is entailed by tool output",
    },
    {
        "method_family": "assignment baseline",
        "example_methods": "lettucedetect_real",
        "needs_training": "no for pretrained version",
        "needs_generation_trace": "no",
        "inference_cost": "medium",
        "role_in_evaluation": "span-detection baseline slot",
    },
    {
        "method_family": "attention-style baseline",
        "example_methods": "attention_lookback_lens_adapted",
        "needs_training": "no",
        "needs_generation_trace": "yes for full method",
        "inference_cost": "medium/high",
        "role_in_evaluation": "approximates context-reliance signal; limitation of this notebook",
    },
    {
        "method_family": "improved method",
        "example_methods": "modernbert_token_classifier",
        "needs_training": "yes",
        "needs_generation_trace": "no",
        "inference_cost": "medium after training",
        "role_in_evaluation": "learns answer-token hallucination spans directly",
    },
]
practicality_df = pd.DataFrame(practicality_rows)
display(practicality_df)


,method_family,example_methods,needs_training,needs_generation_trace,inference_cost,role_in_evaluation
0,trivial baseline,"always_clean, always_hallucinated, random",no,no,very low,checks whether metrics are misleading under cl...
1,rule baseline,keyword_tool_action,no,no,very low,simple sanity check for missing-tool/action ph...
2,semantic verifier,nli_sentence_verifier,no,no,medium,checks whether answer is entailed by tool output
3,assignment baseline,lettucedetect_real,no for pretrained version,no,medium,span-detection baseline slot
4,attention-style baseline,attention_lookback_lens_adapted,no,yes for full method,medium/high,approximates context-reliance signal; limitati...
5,improved method,modernbert_token_classifier,yes,no,medium after training,learns answer-token hallucination spans directly


### Baseline-family summary

**What happens here:** this cell summarizes the best method inside each broad family. It makes the result easier to discuss: the improved learned span detector should be compared not only to individual baselines, but also to the strongest trivial/rule/semantic baseline family.


In [29]:
def method_family(method_name: str) -> str:
    name = str(method_name).lower()
    if "always" in name or "random" in name:
        return "trivial"
    if "keyword" in name:
        return "rule"
    if "nli" in name:
        return "semantic_nli"
    if "lettuce" in name:
        return "lettucedetect_real" if "real" in name else "lettucedetect_slot"
    if "attention_lookback" in name:
        return "lookback_lens_attention_adapted"
    if "lookback" in name:
        return "lookback_lens_proxy"
    if "modernbert" in name or "improved" in name:
        return "improved_learned"
    return "other"

score_col = "sentence_macro_f1" if "sentence_macro_f1" in metrics_df.columns else "sentence_f1"
family_summary = metrics_df.copy()
family_summary["family"] = family_summary["method"].map(method_family)
family_summary = (
    family_summary.sort_values(score_col, ascending=False)
    .groupby("family", as_index=False)
    .first()[["family", "method", score_col, "sentence_balanced_accuracy", "sentence_accuracy"]]
    .sort_values(score_col, ascending=False)
)
display(family_summary)


,family,method,sentence_macro_f1,sentence_balanced_accuracy,sentence_accuracy
0,improved_learned,modernbert_token_classifier,0.785698,0.861111,0.808333
5,semantic_nli,nli_sentence_verifier,0.646316,0.633333,0.766667
4,rule,keyword_tool_action,0.569964,0.583333,0.641667
3,lookback_lens_proxy,lookback_lens_style_proxy,0.549962,0.561111,0.625000
1,lettucedetect_real,lettucedetect_real,0.531511,0.550000,0.591667
2,lookback_lens_attention_adapted,attention_lookback_lens_adapted,0.512500,0.533333,0.566667
6,trivial,always_hallucinated,0.428571,0.500000,0.750000


### Ablation-style comparison

**Explanation:** summarizes the most important methods as an ablation-style table. This is not a separate training run for every combination, but it clearly shows the contribution of trivial baselines, lexical rules, semantic verification, LookBackLens-style support scoring, and the learned ModernBERT detector.

In [30]:
ablation_cols = [
    "method",
    "sentence_macro_f1",
    "sentence_balanced_accuracy",
    "sentence_f1",
    "span_char_f1",
    "span_relaxed_span_f1",
]
available_ablation_cols = [col for col in ablation_cols if col in metrics_df.columns]
ablation_df = metrics_df[available_ablation_cols].copy()
if "sentence_macro_f1" in ablation_df.columns:
    ablation_df = ablation_df.sort_values("sentence_macro_f1", ascending=False)
display(ablation_df)
ablation_df.to_csv("artifacts/ablation_table.csv", index=False)


,method,sentence_macro_f1,sentence_balanced_accuracy,sentence_f1,span_char_f1,span_relaxed_span_f1
9,modernbert_token_classifier,0.785698,0.861111,0.855346,0.956843,0.777778
7,nli_sentence_verifier,0.646316,0.633333,0.852632,0.211063,0.197232
4,keyword_tool_action,0.569964,0.583333,0.745562,0.223919,0.241071
5,lookback_lens_style_proxy,0.549962,0.561111,0.733728,0.387770,0.360360
8,lettucedetect_real,0.531511,0.550000,0.699387,0.157397,0.101695
6,attention_lookback_lens_adapted,0.512500,0.533333,0.675000,0.106030,0.000000
1,always_hallucinated,0.428571,0.500000,0.857143,0.136498,0.009524
2,random_balanced,0.419195,0.450000,0.791878,0.118986,0.000000
3,random_prior,0.418750,0.422222,0.612500,0.124667,0.012500
0,always_clean,0.200000,0.500000,0.000000,0.000000,0.000000


## 3.10 Per-type results

The main binary metric tells whether an answer is hallucinated at all. The per-type table explains where the model works or fails. For corrupted-only subsets, recall is often the most interpretable sentence-level number; for the clean subset, specificity / negative recall is most important.


### Per-type metric table

**What happens here:** we break down performance by hallucination type. This helps identify whether the method is only solving one easy corruption pattern or generalizes across contradiction, overgeneration, and missing-tool cases.


**Explanation:** displays performance by hallucination type. For corrupted-only subsets, recall and span metrics are usually more interpretable than balanced accuracy.


In [31]:
per_type_df = full_result["per_type_metrics"].copy()
ptype_cols = [
    "corruption_type", "n",
    "sentence_accuracy", "sentence_balanced_accuracy", "sentence_recall", "sentence_specificity",
    "sentence_f1", "sentence_negative_f1", "sentence_macro_f1",
    "span_char_f1", "span_relaxed_span_f1",
]
ptype_cols = [col for col in ptype_cols if col in per_type_df.columns]
display(per_type_df[ptype_cols])


,corruption_type,n,sentence_accuracy,sentence_balanced_accuracy,sentence_recall,sentence_specificity,sentence_f1,sentence_negative_f1,sentence_macro_f1,span_char_f1,span_relaxed_span_f1
0,clean,30,0.966667,0.483333,0.000000,0.966667,0.000000,0.983051,0.491525,0.000000,0.000000
1,missing_tool,30,0.966667,0.483333,0.966667,0.000000,0.983051,0.000000,0.491525,0.980919,0.966667
2,overgeneration,30,0.966667,0.483333,0.966667,0.000000,0.983051,0.000000,0.491525,0.978251,0.920635
3,tool_contradiction,30,0.333333,0.166667,0.333333,0.000000,0.500000,0.000000,0.250000,0.771727,0.263158


### Error analysis examples

This table shows false positives and false negatives for the improved method. Use it to write a qualitative discussion: false positives mean the model is over-sensitive on clean answers; false negatives mean it misses hallucinations.


**Explanation:** collects false positives and false negatives for the improved method. These examples support the qualitative error analysis in the report.


In [32]:
from tool_hallucination_detection.schema import sentence_label

thresholds = full_result.get("thresholds", {})
improved_predictions = full_result["predictions"]["improved"]
improved_candidates = metrics_df[metrics_df["method"].astype(str).str.contains("modernbert|improved", case=False, regex=True)]["method"].tolist()
improved_method_name = improved_candidates[0] if improved_candidates else "improved"
improved_threshold = float(thresholds.get(improved_method_name, 0.5))

def _short(text, n=240):
    text = str(text).replace(chr(10), " ").replace(chr(13), " ")
    return text[:n] + ("..." if len(text) > n else "")

error_rows = []
for record, pred in zip(full_result["test_records"], improved_predictions):
    gold = sentence_label(record)
    score = float(pred.get("score", 0.0))
    predicted = int(score >= improved_threshold)
    if gold == predicted:
        continue
    error_rows.append({
        "error_type": "false_positive" if predicted == 1 else "false_negative",
        "corruption_type": record.get("corruption_type"),
        "score": score,
        "threshold": improved_threshold,
        "query": _short(record.get("query", ""), 180),
        "context": _short(record.get("context", ""), 180),
        "output": _short(record.get("output", ""), 260),
        "gold_spans": record.get("labels", []),
        "pred_spans": pred.get("spans", []),
    })

error_df = pd.DataFrame(error_rows)
if len(error_df):
    display(error_df.sort_values(["error_type", "score"], ascending=[True, False]).head(20))
else:
    print("No sentence-level errors for the improved method on this test split.")


,error_type,corruption_type,score,threshold,query,context,output,gold_spans,pred_spans
21,false_negative,tool_contradiction,0.835197,0.835762,Could you provide details about the UFC 257 ev...,"[{""name"": ""Get UFC Fight Details"", ""results"": ...",Details for the UFC 257 event are as follows: ...,"[{'start': 80, 'end': 90, 'text': '2020-02-15'...",[]
1,false_negative,tool_contradiction,0.802157,0.835762,Hey there! Could you help me convert 500 dolla...,"[{""name"": ""Convert"", ""results"": {""converted_am...",Great! You have 500.82 euros available to spen...,"[{'start': 16, 'end': 22, 'text': '500.82', 'l...",[]
15,false_negative,tool_contradiction,0.795180,0.835762,Could you please find the current time for the...,"[{""name"": ""Get Current Time by IP"", ""results"":...",The current time for the location with IP addr...,"[{'start': 65, 'end': 75, 'text': '2025-04-08'...",[]
19,false_negative,tool_contradiction,0.794537,0.835762,Can you get me the latest tier list for grandm...,"[{""name"": ""Get Champion Tier List"", ""results"":...",Here is the list of leagues for the 2023 seaso...,"[{'start': 123, 'end': 129, 'text': 'Sydney', ...",[]
9,false_negative,tool_contradiction,0.707489,0.835762,Can you tell me who led the NBA in assists dur...,"[{""name"": ""topAssistsBySeason"", ""results"": {""p...",Here is the information you requested: ### Bas...,"[{'start': 460, 'end': 466, 'text': 'Boston', ...",[]
11,false_negative,tool_contradiction,0.632100,0.835762,Could you please provide me with detailed info...,"[{""name"": ""Country Details"", ""results"": {""coun...",Here are some Singapore-themed items currently...,"[{'start': 14, 'end': 23, 'text': 'Singapore',...",[]
4,false_negative,tool_contradiction,0.551585,0.835762,Could you tell me the current top music tracks...,"[{""name"": ""Top Music"", ""results"": {""tracks"": [...","Yes, New York zodiac sign, Aquarius, is in the...","[{'start': 5, 'end': 13, 'text': 'New York', '...",[]
12,false_negative,tool_contradiction,0.510959,0.835762,Can you tell me which apps are currently top-g...,"[{""name"": ""Top Grossing Apps"", ""results"": {""to...",The current top-Chicago apps in Japan are: 1. ...,"[{'start': 16, 'end': 23, 'text': 'Chicago', '...",[]
0,false_negative,tool_contradiction,0.495552,0.835762,Could you help me extract the text from this i...,"[{""name"": ""Get Available OCR Algorithms"", ""res...",The current version of the ROME service we are...,"[{'start': 27, 'end': 31, 'text': 'ROME', 'lab...",[]
20,false_negative,tool_contradiction,0.485117,0.835762,I'm working on a design project and I need som...,"[{""name"": ""Trending Images"", ""results"": {""imag...",Great! Here are the trending images from the U...,"[{'start': 956, 'end': 958, 'text': '12', 'lab...",[]


**Explanation:** prints the key report tables directly in the notebook and also saves a small copy under `artifacts/` for convenience. The notebook output itself is sufficient to read the results.


In [41]:
# Print compact report tables in the notebook output.
print("Main metrics table:")
display(metrics_df[[col for col in [
    "method", "sentence_macro_f1", "sentence_balanced_accuracy", "sentence_f1",
    "sentence_roc_auc", "sentence_pr_auc", "span_char_f1", "span_relaxed_span_f1"
] if col in metrics_df.columns]].sort_values("sentence_macro_f1", ascending=False))

print("Per-type behavior for the improved method:")
display(per_type_df[[col for col in [
    "corruption_type", "n", "sentence_recall", "sentence_specificity", "span_char_f1", "span_relaxed_span_f1"
] if col in per_type_df.columns]])

print("Verdict against baselines:")
display(verdict_df)

# Optional lightweight save for attaching artifacts after the run.
Path("artifacts").mkdir(exist_ok=True)
metrics_df.to_csv("artifacts/all_metrics.csv", index=False)
per_type_df.to_csv("artifacts/per_type_metrics.csv", index=False)
verdict_df.to_csv("artifacts/verdict.csv", index=False)
if "error_df" in globals():
    error_df.to_csv("artifacts/error_examples.csv", index=False)
print("Saved compact copies to artifacts/ for optional submission.")


Main metrics table:


,method,sentence_macro_f1,sentence_balanced_accuracy,sentence_f1,sentence_roc_auc,sentence_pr_auc,span_char_f1,span_relaxed_span_f1
9,modernbert_token_classifier,0.785698,0.861111,0.855346,0.894630,0.968790,0.956843,0.777778
7,nli_sentence_verifier,0.646316,0.633333,0.852632,0.654074,0.809387,0.211063,0.197232
4,keyword_tool_action,0.569964,0.583333,0.745562,0.666296,0.846238,0.223919,0.241071
5,lookback_lens_style_proxy,0.549962,0.561111,0.733728,0.642593,0.839647,0.387770,0.360360
8,lettucedetect_real,0.531511,0.550000,0.699387,0.533704,0.774622,0.157397,0.101695
6,attention_lookback_lens_adapted,0.512500,0.533333,0.675000,0.554630,0.779229,0.106030,0.000000
1,always_hallucinated,0.428571,0.500000,0.857143,0.500000,0.750000,0.136498,0.009524
2,random_balanced,0.419195,0.450000,0.791878,0.465185,0.750402,0.118986,0.000000
3,random_prior,0.418750,0.422222,0.612500,0.371852,0.702970,0.124667,0.012500
0,always_clean,0.200000,0.500000,0.000000,0.500000,0.750000,0.000000,0.000000


Per-type behavior for the improved method:


,corruption_type,n,sentence_recall,sentence_specificity,span_char_f1,span_relaxed_span_f1
0,clean,30,0.000000,0.966667,0.000000,0.000000
1,missing_tool,30,0.966667,0.000000,0.980919,0.966667
2,overgeneration,30,0.966667,0.000000,0.978251,0.920635
3,tool_contradiction,30,0.333333,0.000000,0.771727,0.263158


Verdict against baselines:


,comparison,metric,improved_method,improved_score,baseline_method,baseline_score,delta,beats_baseline
0,improved vs best_trivial,sentence_macro_f1,modernbert_token_classifier,0.785698,always_hallucinated,0.428571,0.357126,True
1,improved vs best_non_improved,sentence_macro_f1,modernbert_token_classifier,0.785698,nli_sentence_verifier,0.646316,0.139382,True
2,improved vs best_assignment_baseline,sentence_macro_f1,modernbert_token_classifier,0.785698,nli_sentence_verifier,0.646316,0.139382,True


Saved compact copies to artifacts/ for optional submission.


**Explanation:** prints a compact sample of model predictions with gold and predicted spans for manual qualitative inspection.


In [35]:
# Inspect several predictions for qualitative analysis.
test_records = full_result["test_records"]
improved_predictions = full_result["predictions"]["improved"]

rows = []
for record, pred in list(zip(test_records, improved_predictions))[:20]:
    rows.append({
        "id": record["id"],
        "type": record["corruption_type"],
        "output": record["output"][:300],
        "gold_spans": record.get("labels", []),
        "pred_spans": pred.get("spans", []),
        "score": pred.get("score", 0.0),
    })

pd.DataFrame(rows)


,id,type,output,gold_spans,pred_spans,score
0,toolace-21::clean,clean,Your generated password is: **H4G8fb9A3eKt2LpO...,[],[],0.206426
1,toolace-21::tool_contradiction,tool_contradiction,Your generated password is: **H4G8fb9A3eKt2LpO...,"[{'start': 123, 'end': 208, 'text': ' However,...","[{'start': 122, 'end': 208, 'text': ' However...",1.000000
2,toolace-21::overgeneration,overgeneration,Your generated password is: **H4G8fb9A3eKt2LpO...,"[{'start': 123, 'end': 197, 'text': ' Users we...","[{'start': 122, 'end': 197, 'text': ' Users w...",0.999999
3,toolace-21::missing_tool,missing_tool,Your generated password is: **H4G8fb9A3eKt2LpO...,"[{'start': 182, 'end': 236, 'text': ' I can se...","[{'start': 182, 'end': 236, 'text': ' I can se...",1.000000
4,toolace-149::clean,clean,The current version of the OCR service we are ...,[],[],0.391398
5,toolace-149::tool_contradiction,tool_contradiction,The current version of the ROME service we are...,"[{'start': 27, 'end': 31, 'text': 'ROME', 'lab...",[],0.495552
6,toolace-149::overgeneration,overgeneration,The current version of the OCR service we are ...,"[{'start': 67, 'end': 153, 'text': ' Historica...","[{'start': 66, 'end': 153, 'text': ' Historic...",0.999992
7,toolace-149::missing_tool,missing_tool,The current version of the OCR service we are ...,"[{'start': 67, 'end': 121, 'text': ' I can sen...","[{'start': 66, 'end': 121, 'text': ' I can se...",1.000000
8,toolace-195::clean,clean,Great! You have 435.5 euros available to spend...,[],[],0.312122
9,toolace-195::tool_contradiction,tool_contradiction,Great! You have 500.82 euros available to spen...,"[{'start': 16, 'end': 22, 'text': '500.82', 'l...",[],0.802157


## 3.12 Report-ready conclusion checklist

Use this checklist after running the notebook. The final report should explicitly mention: (1) macro-F1/balanced accuracy rather than only positive-class F1, (2) no split leakage, (3) strict naming of unavailable/fallback external baselines, (4) weighted token classification, (5) separate sentence/span thresholds, (6) per-type results, and (7) representative false positives/false negatives.


In [40]:
print()
print("Key report artifacts:")
for path in [
    "artifacts/all_metrics.csv",
    "artifacts/per_type_metrics.csv",
    "artifacts/verdict.csv",
    "artifacts/practicality_table.csv",
    "artifacts/baseline_family_summary.csv",
]:
    print("-", path)



Key report artifacts:
- artifacts/all_metrics.csv
- artifacts/per_type_metrics.csv
- artifacts/verdict.csv
- artifacts/practicality_table.csv
- artifacts/baseline_family_summary.csv


## 3.13 Optional: data-size scaling experiment

**Explanation:** optional experiment for the final report. It reruns the pipeline on multiple `max_base_records` values to check whether the quality is stable as the synthetic dataset grows. It is disabled by default because it can be expensive on Colab. Turn it on only after the main run succeeds.

In [37]:
RUN_SCALING_EXPERIMENT = False
SCALING_SIZES = [200, 1000]  # add 3000 if you have enough GPU time

scaling_rows = []
if RUN_SCALING_EXPERIMENT:
    for size in SCALING_SIZES:
        print(f"Running scaling experiment for max_base_records={size}")
        result = evaluate_experiment(
            quick=False,
            max_base_records=size,
            strict_training=False,
            epochs=1.0,
            batch_size=2,
            max_length=1024,
            cache_dir=None,
            use_class_weights=True,
        )
        df = result["all_metrics"].copy()
        improved = df[df["method"].astype(str).str.contains("modernbert|improved", case=False, regex=True)].head(1)
        if len(improved):
            row = improved.iloc[0].to_dict()
            row["max_base_records"] = size
            scaling_rows.append(row)

scaling_df = pd.DataFrame(scaling_rows)
if len(scaling_df):
    display(scaling_df)
    scaling_df.to_csv("artifacts/scaling_experiment.csv", index=False)
else:
    print("Scaling experiment was skipped. Set RUN_SCALING_EXPERIMENT=True to run it.")


Scaling experiment was skipped. Set RUN_SCALING_EXPERIMENT=True to run it.


## 3.14 Optional: publish dataset/model to Hugging Face

The assignment results section expects a published dataset and a model. To publish, upload the generated RAGTruth-like splits (`query`, `context`, `output`, `hallucination_labels`, `corruption_type`, `source_id`) and the trained ModernBERT token classifier with tokenizer/config and threshold values.

Important interpretation: publishing makes the benchmark reproducible, but it does not automatically prove that the dataset is perfect. The dataset is synthetic by assignment design. Its quality should be argued through validation checks, corruption-style diversity, no split leakage, baselines, and error analysis. If time allows, add a short dataset card saying that this is a ToolACE-derived controlled benchmark and that real-world generalization should be tested separately.


**Explanation:** optional publishing step. Turn this on only after the notebook runs successfully and you are ready to upload the dataset/model to Hugging Face.


In [45]:
# from huggingface_hub import notebook_login
# notebook_login()

In [ ]:
# # Publish the constructed dataset and trained model to Hugging Face.
# # Run this cell only after the full experiment has completed successfully.

# from pathlib import Path
# import json
# from datasets import Dataset, DatasetDict
# from huggingface_hub import HfApi, notebook_login, whoami, get_token

# PUSH_TO_HUB = False # for submit should be true

# # Login interactively if needed. You need a Hugging Face token with write access.
# # In Colab this opens a token input box. If you already logged in, this is harmless.
# notebook_login()

# api = HfApi()

# # Use the actual Hugging Face username from the active token.
# # This avoids 401 errors caused by trying to push into a namespace you do not own.
# HF_USERNAME = whoami()["name"]
# HF_TOKEN = get_token()

# HF_DATASET_REPO = f"{HF_USERNAME}/toolace-tool-calling-hallucination-ragtruth"
# HF_MODEL_REPO = f"{HF_USERNAME}/modernbert-tool-calling-hallucination-detector"

# print("HF username:", HF_USERNAME)
# print("Dataset repo:", HF_DATASET_REPO)
# print("Model repo:", HF_MODEL_REPO)

# if HF_TOKEN is None:
#     raise RuntimeError("No Hugging Face token found. Run notebook_login() and paste a write token.")

# def to_assignment_record(record):
#     """Convert internal record format to assignment-facing RAGTruth-like format."""
#     labels = list(record.get("hallucination_labels") or record.get("labels") or [])
#     return {
#         "id": str(record.get("id", "")),
#         "source_id": str(record.get("source_id", "")),
#         "split": str(record.get("split", "")),
#         "corruption_type": str(record.get("corruption_type", "")),
#         "query": str(record.get("query", "")),
#         "context": str(record.get("context", "")),
#         "output": str(record.get("output", "")),
#         "hallucination_labels": labels,
#         # Extra tool-calling fields are useful for reproducibility.
#         "tools": str(record.get("tools", "")),
#         "tool_call": str(record.get("tool_call", "")),
#         # Keep internal alias for convenience, but graders should use hallucination_labels.
#         "labels": labels,
#     }

# source_dataset = full_result.get("dataset", dataset)

# # Main dataset: all clean and corrupted examples with corruption_type column.
# hf_dataset = DatasetDict({
#     split: Dataset.from_list([to_assignment_record(record) for record in records])
#     for split, records in source_dataset.items()
# })

# # Push the main dataset. This is sufficient for evaluation because each row has corruption_type.
# hf_dataset.push_to_hub(
#     HF_DATASET_REPO,
#     token=HF_TOKEN,
#     private=False,
# )
# print("Main dataset pushed to:", f"https://huggingface.co/datasets/{HF_DATASET_REPO}")

# # Also push three per-type configs/subsets to make the assignment's "three datasets" requirement explicit.
# # Each subset contains clean examples plus one hallucination type.
# for hallucination_type in ["tool_contradiction", "overgeneration", "missing_tool"]:
#     per_type_dataset = DatasetDict({
#         split: Dataset.from_list([
#             to_assignment_record(record)
#             for record in records
#             if record.get("corruption_type") in {"clean", hallucination_type}
#         ])
#         for split, records in source_dataset.items()
#     })

#     try:
#         per_type_dataset.push_to_hub(
#             HF_DATASET_REPO,
#             config_name=hallucination_type,
#             token=HF_TOKEN,
#             private=False,
#         )
#         print("Per-type config pushed:", hallucination_type)
#     except TypeError:
#         # Older datasets versions may not support config_name in push_to_hub.
#         fallback_repo = f"{HF_USERNAME}/toolace-tool-calling-hallucination-ragtruth-{hallucination_type.replace('_', '-')}"
#         per_type_dataset.push_to_hub(
#             fallback_repo,
#             token=HF_TOKEN,
#             private=False,
#         )
#         print("Per-type dataset pushed to separate repo:", f"https://huggingface.co/datasets/{fallback_repo}")

# # Upload a dataset card.
# dataset_card = f"""---
# language:
# - en
# tags:
# - hallucination-detection
# - tool-calling
# - ragtruth
# - toolace
# - synthetic-data
# task_categories:
# - text-classification
# pretty_name: ToolACE-derived tool-calling hallucination dataset
# ---

# # ToolACE-derived Tool-Calling Hallucination Dataset

# This dataset was created for the course assignment **Hallucination Detection in Tool Calling**.

# It is synthetic by design: starting from ToolACE-style tool-calling dialogues, we automatically inject three required hallucination types:

# 1. `tool_contradiction`
# 2. `overgeneration`
# 3. `missing_tool`

# Each example follows a RAGTruth-like format:

# - `query`: user query
# - `context`: tool output
# - `output`: final model answer
# - `hallucination_labels`: span-level hallucination labels with character offsets

# Additional fields such as `tools`, `tool_call`, `source_id`, `split`, and `corruption_type` are included for reproducibility and analysis.

# ## Splits

# The dataset contains train, validation, and test splits. Splits are separated by `source_id` to avoid leakage between clean and corrupted variants of the same original dialogue.

# ## Evaluation setup

# Clean examples are negative at sentence level. The three corrupted types are positive. The dataset supports both sentence-level binary hallucination detection and span-level hallucination localization.

# ## Required hallucination types

# The dataset contains the three hallucination categories required by the assignment:

# - `tool_contradiction`: the answer contradicts the tool output;
# - `overgeneration`: the answer adds unsupported information not present in the tool output;
# - `missing_tool`: the answer suggests an action requiring a tool that is not available.

# The main dataset contains all examples with a `corruption_type` column. In addition, per-type dataset configurations are uploaded for each hallucination category.

# ## Limitations

# The dataset is automatically corrupted and may contain artifacts of the corruption procedure. This is expected for the assignment, because the task requires automatic hallucination injection. The dataset is suitable as a controlled benchmark for this project, but real-world generalization should be evaluated separately.
# """

# Path("dataset_card.md").write_text(dataset_card, encoding="utf-8")
# api.upload_file(
#     path_or_fileobj="dataset_card.md",
#     path_in_repo="README.md",
#     repo_id=HF_DATASET_REPO,
#     repo_type="dataset",
#     token=HF_TOKEN,
# )
# print("Dataset card uploaded.")

# # Push trained ModernBERT model folder.
# model_path = full_result.get("model_path")
# if model_path is None:
#     raise ValueError("No model_path found in full_result. The model was not uploaded.")

# model_path = Path(model_path)
# if not model_path.exists():
#     raise FileNotFoundError(f"Model path does not exist: {model_path}")

# # Save useful evaluation metadata next to the model before uploading.
# (model_path / "thresholds.json").write_text(
#     json.dumps(full_result.get("thresholds", {}), indent=2),
#     encoding="utf-8",
# )

# summary = {
#     "main_metric": "sentence_macro_f1",
#     "improved_method": "modernbert_token_classifier",
#     "dataset_repo": HF_DATASET_REPO,
#     "metrics": metrics_df.to_dict(orient="records"),
#     "per_type_metrics": per_type_df.to_dict(orient="records"),
#     "verdict": verdict_df.to_dict(orient="records"),
# }
# (model_path / "evaluation_summary.json").write_text(
#     json.dumps(summary, indent=2),
#     encoding="utf-8",
# )

# model_card = f"""---
# language:
# - en
# tags:
# - hallucination-detection
# - tool-calling
# - token-classification
# - modernbert
# pipeline_tag: token-classification
# ---

# # ModernBERT Tool-Calling Hallucination Detector

# This model is a task-specific ModernBERT token classifier trained on a ToolACE-derived synthetic hallucination dataset.

# Dataset: https://huggingface.co/datasets/{HF_DATASET_REPO}

# The model receives a user query, tool output, and final answer. The training loss is applied only to answer tokens. Tokens overlapping hallucinated spans are labeled as positive; other answer tokens are labeled as negative; query/context tokens are masked.

# ## Intended use

# The model is intended for the course assignment **Hallucination Detection in Tool Calling** and for controlled evaluation on the accompanying synthetic dataset.

# ## Evaluation

# The main evaluation metric is sentence-level macro-F1, with additional balanced accuracy, ROC-AUC, PR-AUC, and span-level character F1. Detailed results are stored in `evaluation_summary.json`.

# ## Limitations

# The model is trained on automatically corrupted examples. It should be interpreted as a detector for this controlled benchmark rather than a general-purpose hallucination detector for all real-world tool-calling systems.
# """

# (model_path / "README.md").write_text(model_card, encoding="utf-8")

# api.upload_folder(
#     folder_path=str(model_path),
#     repo_id=HF_MODEL_REPO,
#     repo_type="model",
#     token=HF_TOKEN,
# )
# print("Model pushed to:", f"https://huggingface.co/{HF_MODEL_REPO}")

# print("Done.")
# print("Dataset:", f"https://huggingface.co/datasets/{HF_DATASET_REPO}")
# print("Model:", f"https://huggingface.co/{HF_MODEL_REPO}")

In [49]:
# from pathlib import Path
# from huggingface_hub import HfApi, whoami, get_token

# api = HfApi()

# HF_USERNAME = whoami()["name"]
# HF_TOKEN = get_token()

# HF_MODEL_REPO = f"{HF_USERNAME}/modernbert-tool-calling-hallucination-detector"

# print("Model repo:", HF_MODEL_REPO)

# model_path = Path(full_result.get("model_path"))
# if not model_path.exists():
#     raise FileNotFoundError(f"Model path does not exist: {model_path}")

# # Create the model repository before uploading files.
# api.create_repo(
#     repo_id=HF_MODEL_REPO,
#     repo_type="model",
#     token=HF_TOKEN,
#     private=False,
#     exist_ok=True,
# )

# # Upload trained model folder.
# api.upload_folder(
#     folder_path=str(model_path),
#     repo_id=HF_MODEL_REPO,
#     repo_type="model",
#     token=HF_TOKEN,
# )

# print("Model pushed to:", f"https://huggingface.co/{HF_MODEL_REPO}")

Model repo: marrita/modernbert-tool-calling-hallucination-detector


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...ckpoint-278/rng_state.pth:  77%|#######7  | 11.3kB / 14.6kB            

  ...eckpoint-278/optimizer.pt:   0%|          | 2.58MB / 1.20GB            

...

  ...eckpoint-278/scheduler.pt:   7%|7         |   107B / 1.47kB            

  ...ssifier/training_args.bin:   7%|7         |   433B / 5.91kB            

  ...int-278/training_args.bin:   7%|7         |   433B / 5.91kB            

  .../checkpoint-278/scaler.pt:   7%|7         |   101B / 1.38kB            

Model pushed to: https://huggingface.co/marrita/modernbert-tool-calling-hallucination-detector


## 3.15 Reproduction summary

The main public API remains simple:

```python
from tool_hallucination_detection import prepare_dataset, evaluate_experiment

dataset = prepare_dataset(quick=False, max_base_records=200, cache_dir=None)
full_result = evaluate_experiment(
    quick=False,
    max_base_records=200,
    strict_training=True,
    use_class_weights=True,
    include_lettucedetect_fallback=False,
    require_real_lettucedetect=True,
    include_attention_lookback=True,
    require_attention_lookback=True,
    cache_dir=None,
)
full_result["all_metrics"]
```

This notebook adds the following checks and improvements:

- harder synthetic corruptions for the three required hallucination types;
- RAGTruth-like `hallucination_labels` with exact character offsets;
- validation of spans and required fields;
- no `source_id` leakage across train/validation/test;
- explicit class-balance inspection;
- corruption-style audit to check that examples are not all the same template;
- trivial baselines (`always_clean`, `always_hallucinated`, `random_balanced`, `random_prior`);
- real LettuceDetect evaluation without silently substituting a keyword fallback;
- adapted attention-based LookBackLens using teacher-forced causal-LM attention maps;
- macro-F1 and balanced accuracy in addition to positive-class F1;
- confusion matrix by method;
- class-weighted token loss for rare hallucinated tokens;
- separate validation-tuned sentence and span thresholds;
- qualitative false-positive / false-negative inspection.

The main limitation remains that this is a synthetic ToolACE-derived benchmark. That is expected by the assignment, but the report should state that scores demonstrate performance on this controlled benchmark, not universal real-world hallucination detection. The adapted LookBackLens baseline is closer to the original method than a lexical proxy because it uses real attention maps, but it is still adapted to fixed answers rather than collected during original autoregressive generation.
